In [1]:
exec('import urllib.request,os,numpy as np,pandas as pd\nU="https://s3.amazonaws.com/macchiato.clue.io/builds/LINCS2020/siginfo_beta.txt"\nif not os.path.exists("siginfo_beta.txt"):\n    print("downloading...");urllib.request.urlretrieve(U,"siginfo_beta.txt")\nh=pd.read_csv("siginfo_beta.txt",sep="\\t",nrows=3)\ncols=[c for c in ("pert_id","pert_type","cell_iname","tas","is_exemplar_sig","qc_pass") if c in h.columns]\ns=pd.read_csv("siginfo_beta.txt",sep="\\t",usecols=cols,low_memory=False)\nprint("rows",len(s),"cols",cols)\ns=s[(s.pert_type=="trt_cp")&(s.is_exemplar_sig==1)&(s.qc_pass==1)&s.tas.notna()]\nprint("qc-passing exemplar compound signatures:",len(s))\nper=s.groupby("pert_id")["tas"].agg(["max","median","count"])\nper.columns=["mx","md","n"]\nprint("distinct compounds:",len(per))\ndef gini(x):\n    x=np.sort(np.asarray(x,float));n=x.size;i=np.arange(1,n+1)\n    return float(2*(i*x).sum()/(n*x.sum())-(n+1)/n)\nprint("SELFTEST gini uniform want 0.3333 got",round(gini(np.random.default_rng(0).uniform(0,1,200000)),4))\nprint("deciles of per-compound MEDIAN tas:",[round(v,4) for v in per.md.quantile(np.arange(0,1.01,.1))])\nprint("gini MEDIAN",round(gini(per.md.values),4),"gini MAX",round(gini(per.mx.values),4))\nprint("rho(max tas, n signatures)",round(per.mx.corr(per.n,method="spearman"),4))\nt=per.md.sum()\nprint("top10pct share",round(per.md.nlargest(max(1,len(per)//10)).sum()/t,4),"top1pct share",round(per.md.nlargest(max(1,len(per)//100)).sum()/t,4))\nper.to_csv("activity_per_compound.csv");print("DONE wrote activity_per_compound.csv")')

downloading...
rows 1201944 cols ['pert_id', 'pert_type', 'cell_iname', 'tas', 'is_exemplar_sig', 'qc_pass']
qc-passing exemplar compound signatures: 90834
distinct compounds: 26425
SELFTEST gini uniform want 0.3333 got 0.3333
deciles of per-compound MEDIAN tas: [0.0111, 0.085, 0.1073, 0.1236, 0.1384, 0.1538, 0.1718, 0.195, 0.229, 0.2933, 0.8585]
gini MEDIAN 0.2744 gini MAX 0.3308
rho(max tas, n signatures) 0.6179
top10pct share 0.2233 top1pct share 0.0352
DONE wrote activity_per_compound.csv


In [2]:
exec('import subprocess,sys,time\nsubprocess.run([sys.executable,"-m","pip","install","-q","fsspec","aiohttp","h5py"],check=False)\nimport fsspec,h5py,numpy as np\nu="https://s3.amazonaws.com/macchiato.clue.io/builds/LINCS2020/level5/level5_beta_trt_cp_n720216x12328.gctx"\nt0=time.time();f=fsspec.open(u,"rb").open();h=h5py.File(f,"r")\nprint("opened remotely in",round(time.time()-t0,1),"s  top keys",list(h.keys()))\nd=h["0"]["DATA"]["0"]["matrix"]\nprint("matrix shape",d.shape,"dtype",d.dtype,"chunks",d.chunks)\nprint("ROW ids",[x.decode() for x in h["0"]["META"]["ROW"]["id"][:3]])\nprint("COL ids",[x.decode() for x in h["0"]["META"]["COL"]["id"][:3]])\nt1=time.time();sub=d[0:50,:];print("sliced",sub.shape,"in",round(time.time()-t1,1),"s  finite",int(np.isfinite(sub).sum()),"mean",round(float(np.nanmean(sub)),4))\nprint("NOTHING WAS DOWNLOADED IN FULL: file is 35.5 GB")')

opened remotely in 1.1 s  top keys ['0']
matrix shape (720216, 12328) dtype float32 chunks None
ROW ids ['10', '100', '1000']
COL ids ['ABY001_A375_XH:BRD-A61304759:0.625:24', 'ABY001_A375_XH:BRD-A61304759:0.625:3', 'ABY001_A375_XH:BRD-A61304759:10:24']
sliced (50, 12328) in 0.1 s  finite 616400 mean 0.0783
NOTHING WAS DOWNLOADED IN FULL: file is 35.5 GB


In [3]:
exec('import time,numpy as np\nnR=h["0"]["META"]["ROW"]["id"].shape[0];nC=h["0"]["META"]["COL"]["id"].shape[0]\nprint("len(META/ROW/id)",nR,"  len(META/COL/id)",nC,"  matrix",d.shape)\nprint("AXIS 0 of the matrix is","COL ids = SIGNATURES" if d.shape[0]==nC else "ROW ids")\nprint("AXIS 1 of the matrix is","ROW ids = GENES" if d.shape[1]==nR else "COL ids")\nt=time.time();blk=d[0:5000,:];el=time.time()-t\nmb=blk.nbytes/1e6\nprint("read 5000 signatures:",round(mb,1),"MB in",round(el,1),"s =",round(mb/el,1),"MB/s")\nfull=d.shape[0]*d.shape[1]*4/1e9\nprint("one full streaming pass over",round(full,1),"GB would take about",round(full*1000/(mb/el)/60,1),"minutes")\nprint("chunks",d.chunks,"-> contiguous, so GENE-wise slicing is strided and slow; SIGNATURE-wise blocks are fast")')

len(META/ROW/id) 12328   len(META/COL/id) 720216   matrix (720216, 12328)
AXIS 0 of the matrix is COL ids = SIGNATURES
AXIS 1 of the matrix is ROW ids = GENES
read 5000 signatures: 246.6 MB in 9.3 s = 26.5 MB/s
one full streaming pass over 35.5 GB would take about 22.3 minutes
chunks None -> contiguous, so GENE-wise slicing is strided and slow; SIGNATURE-wise blocks are fast


In [4]:
exec('import pandas as pd,urllib.request,os\nif not os.path.exists("geneinfo_beta.txt"): urllib.request.urlretrieve("https://s3.amazonaws.com/macchiato.clue.io/builds/LINCS2020/geneinfo_beta.txt","geneinfo_beta.txt")\ng=pd.read_csv("geneinfo_beta.txt",sep="\\t",low_memory=False)\nprint("geneinfo rows",len(g),"cols",list(g.columns))\nfs=[c for c in g.columns if "feature_space" in c or "space" in c]\nprint("feature_space counts:")\nprint(g[fs[0]].value_counts() if fs else "no feature_space column")\nlm=g[g[fs[0]]=="landmark"] if fs else g.iloc[0:0]\nprint("landmark genes",len(lm))\nrid=[x.decode() for x in h["0"]["META"]["ROW"]["id"][:]]\nprint("matrix gene axis",len(rid)," landmark ids present in matrix:",len(set(map(str,lm.gene_id))&set(rid)))\nkeep=90834\nprint("retained matrix would be",keep,"x",len(lm),"float32 =",round(keep*len(lm)*4/1e6,1),"MB")')

geneinfo rows 12328 cols ['gene_id', 'gene_symbol', 'ensembl_id', 'gene_title', 'gene_type', 'src', 'feature_space']
feature_space counts:
feature_space
best inferred    9196
inferred         2154
landmark          978
Name: count, dtype: int64
landmark genes 978
matrix gene axis 12328  landmark ids present in matrix: 978
retained matrix would be 90834 x 978 float32 = 355.3 MB


In [6]:
P1="CmltcG9ydCBnemlwLG9zLHJlLHVybGxpYi5yZXF1ZXN0CkdTRT0iR1NFNTczMzgiCkNUUkw9W3IiXGJub25bLVxzXT9mYWlsaW5nXGIiLHIiXGJub25mYWlsaW5nXGIiLHIiXGJjb250cm9sXGIiLHIiXGJkb25vclxiIixyIlxiaGVhbHRoeVxiIl0KQ0FTRT1bciJcYmlzY2hlbWljXGIiLHIiXGJkaWxhdGVkXGIiLHIiXGJjYXJkaW9teW9wYXRoeVxiIixyIlxiZmFpbGluZ1xiIixyIlxiaGVhcnQgZmFpbHVyZVxiIl0KcD1HU0UrIl9zZXJpZXNfbWF0cml4LnR4dCIKaWYgbm90IG9zLnBhdGguZXhpc3RzKHApOgogICAgdT0iaHR0cHM6Ly9mdHAubmNiaS5ubG0ubmloLmdvdi9nZW8vc2VyaWVzLyIrR1NFWzotM10rIm5ubi8iK0dTRSsiL21hdHJpeC8iK0dTRSsiX3Nlcmllc19tYXRyaXgudHh0Lmd6IgogICAgcHJpbnQoImRvd25sb2FkaW5nIix1KQogICAgb3BlbihwLCJ3IixlbmNvZGluZz0idXRmLTgiKS53cml0ZShnemlwLmRlY29tcHJlc3ModXJsbGliLnJlcXVlc3QudXJsb3Blbih1LHRpbWVvdXQ9MzAwKS5yZWFkKCkpLmRlY29kZSgidXRmLTgiLCJyZXBsYWNlIikpCmFjYz1bXTt0aXQ9W107Y2g9W10KZm"
P2="9yIGxpbmUgaW4gb3BlbihwLGVuY29kaW5nPSJ1dGYtOCIsZXJyb3JzPSJyZXBsYWNlIik6CiAgICBpZiBsaW5lLnN0YXJ0c3dpdGgoIiFzZXJpZXNfbWF0cml4X3RhYmxlX2JlZ2luIik6IGJyZWFrCiAgICBpZiBub3QgbGluZS5zdGFydHN3aXRoKCIhU2FtcGxlXyIpOiBjb250aW51ZQogICAgayxfLHJlc3Q9bGluZS5wYXJ0aXRpb24oIlx0Iik7IHY9W3guc3RyaXAoKS5zdHJpcChjaHIoMzQpKSBmb3IgeCBpbiByZXN0LnJzdHJpcCgiXG4iKS5zcGxpdCgiXHQiKV0KICAgIGtsPWsubG93ZXIoKQogICAgaWYga2w9PSIhc2FtcGxlX2dlb19hY2Nlc3Npb24iOiBhY2M9dgogICAgZWxpZiBrbD09IiFzYW1wbGVfdGl0bGUiOiB0aXQ9dgogICAgZWxpZiBrbC5zdGFydHN3aXRoKCIhc2FtcGxlX2NoYXJhY3RlcmlzdGljcyIpOiBjaC5hcHBlbmQodikKbWV0YT17fQpmb3IgaSxnIGluIGVudW1lcmF0ZShhY2MpOgogICAgZj0oW3RpdFtpXV0gaWYgaTxsZW4odGl0KSBlbHNlIFtdKStbcltpXSBmb3IgciBpbiBjaCBpZiBpPGxlbihyKSBhbmQgcltpXV0KICAgIG1ldGFbZ109ZgpkZWYgY2xzKGYpOgogICAgYj0iIDsg"
P3="Ii5qb2luKGYpLmxvd2VyKCkKICAgIGlmIGFueShyZS5zZWFyY2goeCxiKSBmb3IgeCBpbiBDVFJMKTogcmV0dXJuICJjb250cm9sIgogICAgaWYgYW55KHJlLnNlYXJjaCh4LGIpIGZvciB4IGluIENBU0UpOiByZXR1cm4gImNhc2UiCiAgICByZXR1cm4gTm9uZQpyb3M9e2c6Y2xzKGYpIGZvciBnLGYgaW4gbWV0YS5pdGVtcygpfQpjYT1bZyBmb3IgZyBpbiByb3MgaWYgcm9zW2ddPT0iY2FzZSJdOyBjbz1bZyBmb3IgZyBpbiByb3MgaWYgcm9zW2ddPT0iY29udHJvbCJdOyB1bj1bZyBmb3IgZyBpbiByb3MgaWYgcm9zW2ddIGlzIE5vbmVdCnByaW50KCI9Iio2Nik7cHJpbnQoIlBST1BPU0VEIFJPU1RFUiAgLS0gY29udHJvbCBwYXR0ZXJucyBhcmUgbWF0Y2hlZCBCRUZPUkUgY2FzZSIpO3ByaW50KCI9Iio2NikKcHJpbnQoImNhc2VzICVkICAgY29udHJvbHMgJWQgIChoYWx2ZXMgb2YgJWQpICAgdW5hc3NpZ25lZCAlZCAgIHRvdGFsICVkIiUobGVuKGNhKSxsZW4oY28pLGxlbihjbykvLzIsbGVuKHVuKSxsZW4obWV0YSkpKQpmb3IgbGFiLGlkcyBpbiAoKCJDQVNFIixjYSksKCJDT05UUk9MIixjbyksKCJVTk"
P4="FTU0lHTkVEIix1bikpOgogICAgaWYgaWRzOgogICAgICAgIHByaW50KCJcbiIrbGFiKyIsIGZpcnN0IDYgb2YgJWQ6IiVsZW4oaWRzKSkKICAgICAgICBmb3IgZyBpbiBpZHNbOjZdOiBwcmludCgiICAiLGcsIiA7ICIuam9pbihtZXRhW2ddKVs6OTBdKQppZiBsZW4oY28pPDIwOiBwcmludCgiXG5GRVdFUiBUSEFOIDIwIENPTlRST0xTIC0gc3BsaXRzIHN0b3AgYmVpbmcgaW5kZXBlbmRlbnQgZHJhd3MiKQppZiB1bjogcHJpbnQoIlxuJWQgdW5hc3NpZ25lZCBhcmUgRVhDTFVERUQsIG5vdCBkZWZhdWx0ZWQuIElmIHRoYXQgaXMgbGFyZ2UgdGhlIHJ1bGVzIGFyZSB3cm9uZy4iJWxlbih1bikpCmltcG9ydCBjc3YKdz1jc3Yud3JpdGVyKG9wZW4oR1NFKyJfcm9zdGVyLmNzdiIsInciLG5ld2xpbmU9IiIsZW5jb2Rpbmc9InV0Zi04IikpOyB3LndyaXRlcm93KFsiZ3NtIiwiZ3JvdXAiLCJjaGFyYWN0ZXJpc3RpY3MiXSkKZm9yIGcgaW4gcm9zOiB3LndyaXRlcm93KFtnLHJvc1tnXSBvciAidW5hc3NpZ25lZCIsIiA7ICIuam9pbihtZXRhW2ddKV0pCnByaW50KCJcbndyb3RlIixHU0UrIl9yb3N0ZXIuY3N2IikK"

In [7]:
import base64,hashlib
B=P1+P2+P3+P4
print("len",len(B),"want",3064)
print("md5",hashlib.md5(B.encode()).hexdigest(),"want","cdc044311ff740656a6ecad5b6191c3e")
assert len(B)==3064 and hashlib.md5(B.encode()).hexdigest()=="cdc044311ff740656a6ecad5b6191c3e","TRANSFER CORRUPTED - do not run"
exec(base64.b64decode(B))

len 3064 want 3064
md5 cdc044311ff740656a6ecad5b6191c3e want cdc044311ff740656a6ecad5b6191c3e
downloading https://ftp.ncbi.nlm.nih.gov/geo/series/GSE57nnn/GSE57338/matrix/GSE57338_series_matrix.txt.gz
PROPOSED ROSTER  -- control patterns are matched BEFORE case
cases 177   controls 136  (halves of 68)   unassigned 0   total 313

CASE, first 6 of 177:
   GSM1379812 C02065 Idiopathic Dilated CMP Left Ventricle ; tissue: heart left ventricle ; heart failur
   GSM1379813 C02066 Ischemic Left Ventricle ; tissue: heart left ventricle ; heart failure: yes ; disea
   GSM1379814 C02070 Ischemic Left Ventricle ; tissue: heart left ventricle ; heart failure: yes ; disea
   GSM1379815 C02071 Ischemic Left Ventricle ; tissue: heart left ventricle ; heart failure: yes ; disea
   GSM1379816 C02085 Idiopathic Dilated CMP Left Ventricle ; tissue: heart left ventricle ; heart failur
   GSM1379817 C02086 Idiopathic Dilated CMP Left Ventricle ; tissue: heart left ventricle ; heart failur

CONTROL, first 6

In [8]:
Q1="aW1wb3J0IG9zLHJlLGd6aXAsdXJsbGliLnJlcXVlc3QsbnVtcHkgYXMgbnAscGFuZGFzIGFzIHBkCkdTRT0iR1NFNTczMzgiOyBTTT1HU0UrIl9zZXJpZXNfbWF0cml4LnR4dCIKcGxhdD1Ob25lOyBucHJvYmU9MApmb3IgbGluZSBpbiBvcGVuKFNNLGVuY29kaW5nPSJ1dGYtOCIsZXJyb3JzPSJyZXBsYWNlIik6CiAgICBpZiBsaW5lLnN0YXJ0c3dpdGgoIiFTZXJpZXNfcGxhdGZvcm1faWQiKTogcGxhdD1saW5lLnNwbGl0KCJcdCIpWzFdLnN0cmlwKCkuc3RyaXAoY2hyKDM0KSkKICAgIGlmIGxpbmUuc3RhcnRzd2l0aCgiIXNlcmllc19tYXRyaXhfdGFibGVfYmVnaW4iKTogYnJlYWsKcHJpbnQoInBsYXRmb3JtIixwbGF0KQpoZHI9Tm9uZTsgcm93cz1bXQppbmI9RmFsc2UKZm9yIGxpbmUgaW4gb3BlbihTTSxlbmNvZGluZz0idXRmLTgiLGVycm9ycz0icmVwbGFjZSIpOgogICAgaWYgbGluZS5zdGFydHN3aXRoKCIhc2VyaWVzX21hdHJpeF90YWJsZV9lbmQiKTogYnJlYWsKICAgIGlmIGluYjoK"
Q2="ICAgICAgICBwPWxpbmUucnN0cmlwKCJcbiIpLnNwbGl0KCJcdCIpCiAgICAgICAgaWYgaGRyIGlzIE5vbmU6IGhkcj1beC5zdHJpcCgpLnN0cmlwKGNocigzNCkpIGZvciB4IGluIHBdCiAgICAgICAgZWxzZTogcm93cy5hcHBlbmQocCkKICAgIGlmIGxpbmUuc3RhcnRzd2l0aCgiIXNlcmllc19tYXRyaXhfdGFibGVfYmVnaW4iKTogaW5iPVRydWUKcHJpbnQoImV4cHJlc3Npb24gdGFibGU6IixsZW4ocm93cyksInByb2JlcyB4IixsZW4oaGRyKS0xLCJzYW1wbGVzIikKcHJvYmVzPVtyWzBdLnN0cmlwKCkuc3RyaXAoY2hyKDM0KSkgZm9yIHIgaW4gcm93c10KcHJpbnQoInByb2JlIGV4YW1wbGVzIixwcm9iZXNbOjRdKQojIHBsYXRmb3JtIGFubm90YXRpb24KYW5uPXBsYXQrIi5hbm5vdC5neiIKaWYgbm90IG9zLnBhdGguZXhpc3RzKGFubik6CiAgICB1PSJodHRwczovL2Z0cC5uY2JpLm5sbS5uaWguZ292L2dlby9wbGF0Zm9ybXMvIitwbGF0WzotM10rIm5ubi8iK3BsYXQrIi9hbm5vdC8iK3Bs"
Q3="YXQrIi5hbm5vdC5neiIKICAgIHByaW50KCJmZXRjaGluZyIsdSkKICAgIHRyeTogdXJsbGliLnJlcXVlc3QudXJscmV0cmlldmUodSxhbm4pCiAgICBleGNlcHQgRXhjZXB0aW9uIGFzIGU6IHByaW50KCJBTk5PVCBGRVRDSCBGQUlMRUQ6IixlKTsgYW5uPU5vbmUKcDJlPXt9CmlmIGFubiBhbmQgb3MucGF0aC5leGlzdHMoYW5uKToKICAgIGluYj1GYWxzZTsgY29scz1Ob25lCiAgICBmb3IgcmF3IGluIGd6aXAub3Blbihhbm4sInJ0IixlbmNvZGluZz0idXRmLTgiLGVycm9ycz0icmVwbGFjZSIpOgogICAgICAgIGlmIHJhdy5zdGFydHN3aXRoKCIhcGxhdGZvcm1fdGFibGVfZW5kIik6IGJyZWFrCiAgICAgICAgaWYgaW5iOgogICAgICAgICAgICBwPXJhdy5yc3RyaXAoIlxuIikuc3BsaXQoIlx0IikKICAgICAgICAgICAgaWYgY29scyBpcyBOb25lOgogICAgICAgICAgICAgICAgY29scz1wCiAgICAgICAgICAgICAgICBpaWQ9MAogICAgICAgICAgICAgICAgaWdlPVtpIGZvciBpLGMgaW4gZW51"
Q4="bWVyYXRlKGNvbHMpIGlmICJHZW5lIElEIiBpbiBjXQogICAgICAgICAgICAgICAgcHJpbnQoImFubm90IGNvbHMiLGNvbHNbOjZdLCJHZW5lSUQgY29sIGlkeCIsaWdlKQogICAgICAgICAgICBlbHNlOgogICAgICAgICAgICAgICAgaWYgaWdlIGFuZCBsZW4ocCk+aWdlWzBdOgogICAgICAgICAgICAgICAgICAgIGc9cFtpZ2VbMF1dLnNwbGl0KCIvLy8iKVswXS5zdHJpcCgpCiAgICAgICAgICAgICAgICAgICAgaWYgZy5pc2RpZ2l0KCk6IHAyZVtwWzBdXT1pbnQoZykKICAgICAgICBpZiByYXcuc3RhcnRzd2l0aCgiIXBsYXRmb3JtX3RhYmxlX2JlZ2luIik6IGluYj1UcnVlCnByaW50KCJwcm9iZXMgbWFwcGVkIHRvIEVudHJlejoiLGxlbihwMmUpLCJvZiIsbGVuKHByb2JlcykpCiMgbGFuZG1hcmtzCmdpPSJnZW5laW5mb19iZXRhLnR4dCIKaWYgbm90IG9zLnBhdGguZXhpc3RzKGdpKToKICAgIHVybGxpYi5yZXF1ZXN0LnVybHJldHJpZXZlKCJodHRwczovL3MzLmFtYXpvbmF3cy5jb20vbWFj"
Q5="Y2hpYXRvLmNsdWUuaW8vYnVpbGRzL0xJTkNTMjAyMC9nZW5laW5mb19iZXRhLnR4dCIsZ2kpCmc9cGQucmVhZF9jc3YoZ2ksc2VwPSJcdCIpCmxtPXNldChnLmxvY1tnLmZlYXR1cmVfc3BhY2U9PSJsYW5kbWFyayIsImdlbmVfaWQiXS5hc3R5cGUoaW50KSkKcHJpbnQoImxhbmRtYXJrIGdlbmVzOiIsbGVuKGxtKSkKbWFwcGVkPXNldChwMmUuZ2V0KHApIGZvciBwIGluIHByb2Jlcykte05vbmV9Cm92PWxtJm1hcHBlZApwcmludCgiPT09IExBTkRNQVJLIENPVkVSQUdFID09PSIpCnByaW50KCJsYW5kbWFya3MgbWVhc3VyZWQgb24gdGhpcyBwbGF0Zm9ybToiLGxlbihvdiksIm9mIixsZW4obG0pLAogICAgICAiPSAlLjFmJSUiJSgxMDAqbGVuKG92KS9sZW4obG0pKSkKcHJpbnQoImxhbmRtYXJrcyBNSVNTSU5HOiIsbGVuKGxtLW92KSkKbnAuc2F2ZSgibGFuZG1hcmtfb3ZlcmxhcC5ucHkiLG5wLmFycmF5KHNvcnRlZChvdikpKQpwcmludCgid3JvdGUgbGFuZG1hcmtfb3ZlcmxhcC5ucHkiKQo="

In [9]:
import base64,hashlib
B=Q1+Q2+Q3+Q4+Q5
print("len",len(B),"want",3400)
print("md5",hashlib.md5(B.encode()).hexdigest(),"want","fda94d1a5b6704ea8dce271e6ac0184e")
assert len(B)==3400 and hashlib.md5(B.encode()).hexdigest()=="fda94d1a5b6704ea8dce271e6ac0184e","TRANSFER CORRUPTED"
exec(base64.b64decode(B))

len 3400 want 3400
md5 fda94d1a5b6704ea8dce271e6ac0184e want fda94d1a5b6704ea8dce271e6ac0184e
platform GPL11532
expression table: 33297 probes x 313 samples
probe examples ['7892501', '7892502', '7892503', '7892504']
fetching https://ftp.ncbi.nlm.nih.gov/geo/platforms/GPL11nnn/GPL11532/annot/GPL11532.annot.gz
annot cols ['ID', 'Gene title', 'Gene symbol', 'Gene ID', 'UniGene title', 'UniGene symbol'] GeneID col idx [3, 6]
probes mapped to Entrez: 22170 of 33297
landmark genes: 978
=== LANDMARK COVERAGE ===
landmarks measured on this platform: 951 of 978 = 97.2%
landmarks MISSING: 27
wrote landmark_overlap.npy


In [17]:
P0="H4sIACPAvWoC/6VabVPjthb+nl+hej+Q9AaTNwKbHXqHC2HLXApckt12hjIZxVYSF0dK/RJIGf77fY4kJ3II251p2iWxLB095+i82x9+OMjT5GAcyQMhl2yxymZKtiue56V5shSrUSqSSKT+YsX299lg2L9lN9d9piYsmwn2Zy6SFVtECxFHUvjsSqlHNhYTlQj2lERZJKcsyWOsr1TO1ZOMFQ9TJngwwywZzOY8eWRmBzbnWRI9My5Ddnt3eT0csOHPfXZ+ORheXp8N2dnPp3enZ8P+HQ2cVb6eXn3pD1iUMR5kOY/jFQuUzHgkU59dYjRNo6lMmVTZDCjqLIhpaEI72TG9FaEUacUOAeavsxVYi4DyOUqzFISwScoAnqWZWPh6m0TFrdFixlNxyp5U8ihC8BPwPBV6coAbB3ZeZcGzTCSA8iSsVDIh2SRRc/Z50D88areP9wr6PAOlUBB0n12oRAsZ0AR+PYl4KSpWWFKNVbhiQMBWImMx5I6VPNMLlirg4zzmycpnv9pTWKOYREmaEeuVJJeSbmHJHFzkMTHBprmAnEhiJB4OwApzponKF1aocyEztkhUmAdAwitBLLjcJwhETebzMcGdAcwcN9bi9tmQxiY8inMIgqDzOBFcs7FYCAkGsqcoEJVImhPAHn+IIOsBxSR6zmiVJgvNUSHNBq8pnwuLEWcLFiQNQC2eQJ/2h0hUBUecFQwlIhZLLgOxn6qE5C1SwRNoZJqPU8iSINGpc5z0fBGLDCcA3a5UBqrHjFEYIdaN8mgYWsnNoT4RRKBX0gAMEiFknZT5ulDCFagxfIy1sR2Wlop4QqBpFvsAETIpMtK0b67b+nxg4drmiHczn8QeW6P7JjVoZ7PRbnfW1CCGYh38QyWaLyBANv0rWtSZSuuQXJ2lK/zIkziOxn4i4B/SrKLFEqg4xmFGChphV56pXEIp6+wmCWEb4XkUZJXKB7D/zz5G7gboB3YD1AtopPFVpMCcVAkeYaZSWOJ4pedD2TIFkLBFOsyeIQK9jQWDKU2JkQ8skqGAqoZkAiHPOCnMxNppop78tUmTBhTqHZLg6AAwFskgzkl5lYxXIKgV7fZmcDm8/NpnZzfXw7ubK0sS06E90SIj10t+ZWPYROpR4nQhccUi7Y1BzapnqIyXY+lMPbE9qeQ+mR2Y2dM4CG3AkzBS85WCZ5hBUvVCW2k/UB8ncCnSrwzgceFoT9xDqt5XWNUrWPXqjIU0CgGceM3OQfMQQ1rcJ95sMsIeerN8TjPxkSftZrtO3oRjBk8SvsId4BUn3jlFFzgA6z1Z6xObkd+EnDQ+zSDOaKFSOLalKCZ6tVrdgmo1O2290wZU2+5sQQUKDgNSHC1ijiF9U54cNvG9E5RLvHW8TbxTJi5FnqgxDD1Tc77muHncfYdjrVxJEVFh5J9Iidofuw06BqxstJz9m0edbucIizf7d8v7z9NRIFM7Zj/ypNVsvLN/ocYQSgIDFeEn6A+7vhkyCme0Zfu40e24GFqN1lH51I/KGGI+H8W5nLoogOHwPQFTgCD7qbNWo8gHMpVx91i1L9rStYbZwO46hYJHS15iHbt2jt7h/HOLZTzHMUH8iUgV4mtYJxh0lSGHCZ3t24eNxhbTzVZp+zFCYFiWu1b15jvbZ1Ga5kLbGq2ss0nMp1O4hkhajwTFn7oSaHe63Vbp7JvtsgSWs3AEJ6bkBgcUu/P3cm8evi/3zsfOcXnXTmnX4DEcPUahFCt312t4vfrOXacxiZz82CdmNzjskBy0AeTwcCqiwJBmMHEe6zjMJSKH"
P1="g6p7ZA/fQXVUQrVAeI94GqWjFImJuQVt6L4ni9122DpqH1k7bLpC6eK/LW04Lm3PQ2ff7xBK/7ezqy/nfW37UNm23r3baLYIdKOQDo8pi+Pk+VyndNg9bH8sHdFx2SAXfLbLIFsbpzC4PR1enl5t3EI0jaAILEV8oENAfJzoEIIJQiCjWKQiDxUOa53xc2mS8AS+rySr1sdms6xAH8sKpJZROCJTXKSrtcs8Ou5s0PX/Z7y0xkYm2vz4iVkLmiFHS9QUGaTKEU6DRKX4MtQQwpFHaTQPtUolFBOr5yNTb4yQq1Snqaj1dC6UQv0Q6zBw39tvP7B/MU9K6VGEjYXUE9lPrMtEDN+IK70oEUgoJKtOvFmWLdLewcEkW/gyGEe+jOe+jGb+VC0PpkIdmL0PXmif14MXUHj1Kq7DmHgHBpi5OSqB9bPnzJ/+5VlGJiILZoQJuQxKKgjG9ywfFNTBh0Iih1/+HzCoqp4DL+O9QxlkaSlYLZaZEqhKvy1dTTuBdYJZhgBBGb+mG7IXmvZqiThiUciXNIk68xIcos7ese7Ey7PJPumpSBKVpDhWgXgcQLF9ypuqhhLOB4zsPrNKGU6R7RKoF8wpwDxFEEY5J/VxqYHhG5lPNIfqZCftRqNGmUayYTbhT9g+cRFl4jkjHUHe64eCnBPCRVrFzJq+DkX1fdY2gFy5PO2Qi0Yy2SCZ+LrcqNL225yjAEIhYk6AseoL6aqed9AU3Z7fnLyyX/5Ts7vbc6H7Ro8WPEnFaAYO4XPNOrMtsnzoyrzH7nVtlT68skWcG0+ElHqfin6k0TxBDY4Dgr8OKJogl3wUyCIH/VvU7MO+rXbOShNhoXAZeh5l55pUUUPzIhapCdXKi5wKcNxfcnj0cWysbr2qqv82cCRhhFKOzEGPNDFiPESd+b5fQ0UdUxW8KXpjFACmBnqaRagAC/rkbfVI3abLpqITz4Afr0yFt6n6yB36hbz0Nyx9xIOAFCuLKbUmGY0IFH6nSQBg9w91559eRUm/Bh6Zw/HTRYz1tKrqmB+5Imq2pBlPspRUqer9ULaPjJgYjQUSIs9ZSZ8xFPnRpUVlwlt6A06l72h7NXn3SOZiPfgoVnU2quuECVxpQlAnCtxKVr3fM8cfLHmcEudLbIUqo1qz33veXk0zvyTOiZLh3Cx/2OyFxdjPj9WTSKo1lwncOmGQgkFtxU99DOQMvfV5YD1hWC8U8Zul+sBoifmY8/uOdanKk0CMqPkwCmZNUDDHvGPd1sGZ9WUjQrzqOTrj6/ZIWCViax9N52b5cr0VLIB95XEu+uR5qp5UrDjMkliMpk1QgYdlt/BGd8GIq8A2hppmAYzSxJ+5yHidOa7j78hANjvcjl4qSZEo2BoSZV/3u2TMyCw1BMIee5GvHiK19oSsWhTyMNeFrmfYC6G735N7D681HcrX1yaOU31XiHVziyxfsh9ONkO7QuCPP6Jo/3I9pEbl6ee7PkrlXy+HP+ve5e3dzfDm7OaqZ5ES0Pq60QA2VqmLzWflVICSgXOlDxprAqGbBqYtQAlQDlOMHd+EAAPBhj4gbSK5kf4u4IVW6Bl1pDRIfE3c0Etqrz3HdHU/QvslffO+133oObTo85Ld9z62Hl43e+PENxuTMdh+TxU3at9AtGVLdQMrqL2Sf4f7CbJtaEsEHN008OcqzVAAzefwPs1W7Q3Ix95Ph6/P+LG87x1312jfgtgySALSs0DWOgxE9F1Ni8hKWCJTQQKNkDnKHI6Q7SzZLZGl4/9o8YQtd0mIGat9iV57b4VCviZ3wPyNcDpb"
P2="rv37BFUEIL03MuFm51tUEHW1GVms+83OK5tTpYCQ/SgQSXkRUShxB1ZKKox1m26uLkYwy7SfYz4WMbH3T3uEjNqr+9RfTSsXl78Nv9z1cRjwLZ7JlO8rbK9kHwhE1xejRpMi0uXZL6NGy7NDbW+v7swu+VhM+Tz4RS/Cd8t+b63Y0nbMicUEhylQR0UBgtH3jJQo7lBdrLC5UY853UDPHY9SpPFzZHClzuDKe2/td+xpsq8eu/rqfevKUHo3hdG3L89Hd/2L3zMSqP7b0n/be25dZ3rmRbKkKGEYJjZboRlgMHiskqjrpHhQLwSdhFODb6PG4DFWASpe9bhlglWo9e3pYIBEj4yAKNgAwtjF6eUVxmuIQ/oJhA1HL3qDVz1f/9yKOE4wJ3I9A/oCgV44jqn8LM7RXmuVuwPt2xhrdd0sMsLw9MOAUkBFLUL2SqGXcpw2lYzy5KUYK1yBXW9TpPVSGyGaDzo/ssYCVFlSRJbSckf9y9tTkLDb01q6NAs/mNh6c4mge3PB/tvv315ef2ZXl9eIvUXJ0aMGznZpYhKPeY5UFTnYiqb4Lpad00uVjPFHE1NGrKFqYWuwLV1eb8a2A4TdyBYstqdjrWtdfYCq5xqadt5E7r7xgP+NPJyB2lvazXU3zXRIHNLMg8VtSDa3STYdkh/oydumZhrnU/uccM4fBbNc2MeubKyy2SfKVJbmkZh+fBVwqZ/4mfAIUroA0HHJRkmjqW74e3DZcbbXBKkVB9mEgto9FFxhZ0T2/AanP4+e16xC5nUnNHG5qhacO9GW1tbMg0Ga4Ip8e1ZJyl/urohxiC3Xz7RIw9+0KDZPSGo+8neT8q8xeU4/SE+UUh4UK4omUHH9Th+ohOlwX/fsqG/ICB/1mN7Fpfun3wGra1HRfAcUXb6DaSP1b+3s5l0UeMlheqdXVzC1q4v9YX8wHGhf2z/XvhNu0ThO5za5XNyulcoXOG0dDOZQySIQ8GSqC0+uT5TTiaYrVFXJdHnf7D0UDpiXKrP9/aL2xG1v80TW217ec/gtAtAnC8ct1rbDU4n/jUvX5qKfQ4iJ6a9liiU5CvkS0UyFCjxp1nTfAITNo7rtesnEDt0T0+k9FsIv2ce88KWXAwY5X36+HtBDn5/hS31zHt6J9+PRsc1sNSjdt1hnutPUtCpAcMMRlTOYZqD4U5GZ+nDT+v335tkg/bQt8aLP+2/3mdubDPh3+bJ3sgdQr/hFTUymvYitofATVZRtKheD+kIP6x2KYX2xt53W6nse7e+V6hsa6a1LNVy8s1JT9R5IKlWnoa571++m26ggt0D1Ng0w+1LMHMjHgvFxSg+eIXsaIKWaKXjj7bLRlI4mqO/HYiliUxvguG83TXsqI8WfeZTol1feNPCdOlLrW7IqM7C7A7BuSTunJ54DiqF9/UWtB45y9RvSMHYNeWerhaiKmj8y2fGIqh7hCn6jmUWLxGBBDH5v7Q6/4+p5AcRwJ0LHZvbpl9kN9rOulrU5bYpeM+GNhRueTDtRv/5knqFwpKVBpPsxiGUz896M1C+pkMC3Krkp8lUdjswejtRepiXZrNm77v827NnXVNyXkjavA1EX1eraxd3NLzq3su9Wnf7n5mu/Xiap0x96ncNNXEjC9HaPkOsXXPTbBPop/iRK5vZK4/CpfoOgioPRaeJoRP56NPKM4Izzrvwf2PWFZpwmAAA="
import base64,gzip,hashlib
B=P0+P1+P2
print("len",len(B),"want",5304)
print("md5",hashlib.md5(B.encode()).hexdigest())
print("want","7f02e99dce9e3e05acf10a1105c3e314")
assert len(B)==5304 and hashlib.md5(B.encode()).hexdigest()=="7f02e99dce9e3e05acf10a1105c3e314"
S=gzip.decompress(base64.b64decode(B))
assert len(S)==9884 and hashlib.md5(S).hexdigest()=="eb495b75c86b8e0f9215df3648588ef8"
open("survey_series.py","wb").write(S)
compile(S,"survey_series.py","exec")
print("VERIFIED",len(S),"bytes, compiles")

len 5304 want 5304
md5 babbdf613ed0f0802655aa8cc1da0d7c
want 7f02e99dce9e3e05acf10a1105c3e314


AssertionError: 

In [19]:
import hashlib
print(0,len(P0),hashlib.md5(P0.encode()).hexdigest())
print(1,len(P1),hashlib.md5(P1.encode()).hexdigest())
print(2,len(P2),hashlib.md5(P2.encode()).hexdigest())

0 1800 d93b9ab967fae73a0fe16679c1badd89
1 1800 a355e2701d070d1b87e3058afba3ba57
2 1704 58abe49a6233e44cfc3293ae825aa294


In [20]:
P2a="rv37BFUEIL03MuFm51tUEHW1GVms+83OK5tTpYCQ/SgQSXkRUShxB1ZKKox1m26uLkYwy7SfYz4WMbH3T3uEjNqr+9RfTSsXl78Nv9z1cRjwLZ7JlO8rbK9kHwhE1xejRpMi0uXZL6NGy7NDbW+v7swu+VhM+Tz4RS/Cd8t+b63Y0nbMicUEhylQR0UBgtH3jJQo7lBdrLC5UY853UDPHY9SpPFzZHClzuDKe2/td+xpsq8eu/rqfevKUHo3hdG3L89Hd/2L3zMSqP7b0n/be25dZ3rmRbKkKGEYJjZboRlgMHiskqjrpHhQLwSdhFODb6PG4DFWASpe9bhlglWo9e3pYIBEj4yAKNgAwtjF6eUVxmuIQ/oJhA1HL3qDVz1f/9yKOE4wJ3I9A/oCgV44jqn8LM7RXmuVuwPt2xhrdd0sMsLw9MOAUkBFLUL2SqGXcpw2lYzy5KUYK1yBXW9TpPVSGyGaDzo/ssYCVFlSRJbSckf9y9tTkLDb01q6NAs/mNh6c4mge3PB/tvv315ef2ZXl9eIvUXJ0aMGznZpYhKPeY5UFTnYiqb4Lpad00uVjPFHE1NGrKFqYWuwLV1eb8a2A4TdyBYstqdjrWtdfYCq5xqadt5E7r7xgP+NPJyB2lvazXU3zXRIHNLMg8VtSDa3STYdkh/oydumZhrnU/uccM4fBbNc2MeubKyy2SfKVJbmkZh+fBVwqZ/4mfAIUroA0HHJRkmjqW74e3DZcbbXBKkVB9mEgto9FFxhZ0T2/AanP4+e16xC5nUnNHG5qhacO9GW1tbMg0Ga4Ip8e1ZJyl/urohxiC3Xz7RIw9+0KDZPSGo+8neT8q8xeU4/"
P2b="SE+UUh4UK4omUHH9Th+ohOlwX/fsqG/ICB/1mN7Fpfun3wGra1HRfAcUXb6DaSP1b+3s5l0UeMlheqdXVzC1q4v9YX8wHGhf2z/XvhNu0ThO5za5XNyulcoXOG0dDOZQySIQ8GSqC0+uT5TTiaYrVFXJdHnf7D0UDpiXKrP9/aL2xG1v80TW217ec/gtAtAnC8ct1rbDU4n/jUvX5qKfQ4iJ6a9liiU5CvkS0UyFCjxp1nTfAITNo7rtesnEDt0T0+k9FsIv2ce88KWXAwY5X36+HtBDn5/hS31zHt6J9+PRsc1sNSjdt1hnutPUtCpAcMMRlTOYZqD4U5GZ+nDT+v335tkg/bQt8aLP+2/3mdubDPh3+bJ3sgdQr/hFTUymvYitofATVZRtKheD+kIP6x2KYX2xt53W6nse7e+V6hsa6a1LNVy8s1JT9R5IKlWnoa671++m26ggt0D1Ng0w+1LMHMjHgvFxSg+eIXsaIKWaKXjj7bLRlI4mqO/HYiliUxvguG83TXsqI8WfeZTol1feNPCdOlLrW7IqM7C7A7BuSTunJ54DiqF9/UWtB45y9RvSMHYNeWerhaiKmj8y2fGIqh7hCn6jmUWLxGBBDH5v7Q6/4+p5AcRwJ0LHZvbpl9kN9rOulrU5bYpeM+GNhRueTDtRv/5knqFwpKVBpPsxiGUz896M1C+pkMC3Krkp8lUdjswejtRepiXZrNm77v827NnXVNyXkjavA1EX1eraxd3NLzq3su9Wnf7n5mu/Xiap0x96ncNNXEjC9HaPkOsXXPTbBPop/iRK5vZK4/CpfoOgioPRaeJoRP56NPKM4Izzrvwf2PWFZpwmAAA="
import hashlib
print("a",len(P2a),hashlib.md5(P2a.encode()).hexdigest())
print("b",len(P2b),hashlib.md5(P2b.encode()).hexdigest())

a 852 e9a7897c1121581bdd4ba7659fc7a065
b 852 a2b9be45f4bc5cb374dcd5215af6852f


In [21]:
import base64,gzip,hashlib
B=P0+P1+P2a+P2b
print("len",len(B),"want",5304)
print("md5",hashlib.md5(B.encode()).hexdigest())
assert len(B)==5304 and hashlib.md5(B.encode()).hexdigest()=="7f02e99dce9e3e05acf10a1105c3e314"
S=gzip.decompress(base64.b64decode(B))
assert len(S)==9884 and hashlib.md5(S).hexdigest()=="eb495b75c86b8e0f9215df3648588ef8"
open("survey_series.py","wb").write(S)
compile(S,"survey_series.py","exec")
print("VERIFIED",len(S),"bytes, md5 ok, compiles")

len 5304 want 5304
md5 7f02e99dce9e3e05acf10a1105c3e314
VERIFIED 9884 bytes, md5 ok, compiles


In [22]:
!python survey_series.py GSE57338 GSE6710 GSE12027

survey_series self-tests
  PASS  all samples parsed  n=3
  PASS  titles parsed  ['NF_01', 'ICM_02', 'NF_03']
  PASS  source_name parsed  ['left ventricle', 'left ventricle', 'left ventricle']
  PASS  characteristic lines kept SEPARATE, not flattened  2 line(s)
  PASS  line 0 is the disease variable  disease: non-failing
  PASS  line 1 is the tissue variable  tissue: LV
  PASS  flattening test is non-degenerate (flat DOES mix variables)
  PASS  URL construction
  PASS  5-digit GSE URL stub  https://ftp.ncbi.nlm.nih.gov/geo/series/GSE6nnn/GSE6710/matrix/GSE6710_series_matrix.txt.gz

ALL SELF-TESTS PASSED

surveying 3 series. THIS ASSIGNS NOTHING.

GSE57338  row 14/15  query hf_myocardium  assay array
  note: DONE in control 2; here as the survey's positive control
  using cached ./GSE57338_series_matrix.txt

  samples parsed: 313   (protocol expected 313)
  !Sample_title, 6 of 313:
      C02065 Idiopathic Dilated CMP Left Ventricle
      C02066 Ischemic Left Ventricle
      C02070 Ischem

In [23]:
!python survey_series.py GSE2143 GSE2283 GSE174647 GSE10334 GSE35007 GSE134662 GSE104948 GSE16161 GSE256539 GSE162911 2>&1 | tail -220

        159x  Sex: M
        151x  Sex: F
          1x  hb phenotype: SC
    line 4: 6 distinct value(s)
        190x  hb phenotype: SS
         59x  hb phenotype: SC
         30x  hb phenotype: AS
         19x  hb phenotype: A
         12x  hb phenotype: AA
          1x  white blood cells: 13
    line 5: 75 distinct value(s)
         21x  white blood cells: 7
         21x  white blood cells: 10
         20x  white blood cells: 12
         20x  white blood cells: 14
         18x  white blood cells: 8
         18x  white blood cells: 11
         17x  white blood cells: 9
         15x  white blood cells: 6
         15x  white blood cells: 15
         12x  white blood cells: 17
         12x  white blood cells: 16
          9x  white blood cells: 20
          8x  white blood cells: 19
          8x  white blood cells: 13
      ... and 61 more (likely a continuous or per-sample field, not a group label)
    line 6: 199 distinct value(s)
          4x  mean corpuscular volume: 87.1
          4

In [24]:
import glob,collections
P=sorted(glob.glob("*_series_matrix.txt"))
rd=lambda p: open(p,encoding="utf-8",errors="replace").read().split("!series_matrix_table_begin")[0].splitlines()
pick=lambda ls,k: [l.split("\t",1)[1] for l in ls if l.lower().startswith(k)]
vc=lambda s: [(k[:54],n) for k,n in collections.Counter(v.strip().strip(chr(34)) for v in s.split("\t")).most_common(5)]
show=lambda p,tag,k: [print(p[:12],tag,i,vc(s)) for i,s in enumerate(pick(rd(p),k))]
z=[show(p,"TITLE","!sample_title") for p in P]
z=[show(p,"SRC","!sample_source_name") for p in P]
z=[show(p,"CHAR","!sample_characteristics") for p in P]

GSE10334_ser TITLE 0 [('Gingival tissue sample 1 from patient 1, Affected site', 1), ('Gingival tissue sample 2 from patient 1, Affected site', 1), ('Gingival tissue sample 4 from patient 1, Unaffected si', 1), ('Gingival tissue sample 1 from patient 10, Affected sit', 1), ('Gingival tissue sample 2 from patient 10, Affected sit', 1)]
GSE12027_ser TITLE 0 [('LAM_Patient_1', 1), ('LAM_Patient_2', 1), ('LAM_Patient_3', 1), ('LAM_Patient_4', 1), ('LAM_Patient_5', 1)]
GSE134662_se TITLE 0 [('Normal1', 1), ('Normal10', 1), ('Normal2', 1), ('Normal3', 1), ('Normal4', 1)]
GSE174647_se TITLE 0 [('OX_C25_P2C3_001_C1', 1), ('OX_C25_P2C3_001_C2', 1), ('OX_C25_P2C3_001_D1', 1), ('OX_C25_P2C3_001_D2', 1), ('OX_C25_P2C3_001_E1', 1)]
GSE2143_seri TITLE 0 [('1_375', 1), ('2_376', 1), ('2_377', 1), ('2_378', 1), ('2_379', 1)]
GSE256539_se TITLE 0 [('Control-Intima+Media-AH006-Full ROI-AH006-009', 1), ('Control-Intima+Media-AH015-Full ROI-AH015_rescan-001', 1), ('Control-Adventitia-AH015-Full ROI-AH015_

In [25]:
cand=lambda p,tag,k: [print(p[:10],tag,i,c) for i,s in enumerate(pick(rd(p),k)) for c in [vc(s)] if 2<=len(set(v for v in s.split(chr(9))))<=6]
z=[cand(p,"T","!sample_title") for p in P]
z=[cand(p,"S","!sample_source_name") for p in P]
z=[cand(p,"C","!sample_characteristics") for p in P]
print("done")

GSE12027_s S 0 [('Lung_LAM_Nodule', 14), ('PASM, pulmonary-artery smooth muscle (CC-2581, Lonza) ', 7), ('Malme-3M, ATCC, Manassus, VA (HTB-64), human melanoma ', 4)]
GSE134662_ S 0 [('GVHD Colon Biopsie_onset', 22), ('GVHD Colon Biopsie_Refractory', 22), ('Healthy Colon Biopsie', 10)]
GSE256539_ S 0 [('IPAH Lung', 152), ('Control Lung', 68)]
GSE57338_s S 0 [('heart left ventricle, non-failing', 136), ('heart left ventricle, ischemic', 95), ('heart left ventricle, idiopathic dilated CMP', 82)]
GSE6710_se S 0 [('Human skin biopsy from psoriatic patient (lesional ski', 13), ('Human skin biopsy from psoriatic patient (uninvolved s', 13)]
GSE10334_s C 0 [('Gingival tissue phenotype: Diseased (BoP + interproxim', 183), ('Gingival tissue phenotype: Healthy (no BoP + PPD of<5 ', 64)]
GSE12027_s C 0 [('Gender:female', 14), ('PASM, pulmonary-artery smooth muscle (CC-2581, Lonza)', 7), ('Male, Malme-3M, ATCC,Manassus, VA (HTB-64)', 4)]
GSE12027_s C 1 [('Adult', 14), ('', 11)]
GSE12027_s C 2 [('T

In [26]:
import urllib.request,re
exec("def F(g):\n    u='https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=%s&targ=self&form=text&view=brief'%g\n    try:\n        t=urllib.request.urlopen(u,timeout=90).read().decode('utf-8','replace')\n    except Exception as e:\n        print(g,'FETCH-FAIL',str(e)[:70]); return\n    ty=re.findall(r'!Series_type = (.+)',t)\n    ns=len(re.findall(r'!Series_sample_id = (.+)',t))\n    pl=re.findall(r'!Series_platform_id = (.+)',t)\n    rel=[r.strip() for r in re.findall(r'!Series_relation = (.+)',t) if 'Super' in r or 'Sub' in r]\n    print(g,'|',';'.join(ty),'| n=',ns,'| plat=',pl,'| rel=',rel[:4])\n")
G=["GSE2283","GSE3960","GSE104948","GSE104954","GSE16161","GSE6012","GSE162911","GSE159788","GSE2143","GSE12027"]
z=[F(g) for g in G]
print('done')

 | n= 186 | plat= ['GPL1874\r', 'GPL1875\r'] | rel= []
 | n= 102 | plat= ['GPL8300\r'] | rel= []
 | n= 196 | plat= ['GPL22945\r', 'GPL24120\r'] | rel= []
 | n= 195 | plat= ['GPL22945\r', 'GPL24120\r'] | rel= []
 | n= 33 | plat= ['GPL570\r', 'GPL571\r'] | rel= []
 | n= 20 | plat= ['GPL96\r'] | rel= []
 | n= 784 | plat= ['GPL21697\r', 'GPL24676\r', 'GPL29487\r', 'GPL29497\r'] | rel= ['SubSeries of: GSE163530']
 | n= 726 | plat= ['GPL29228\r', 'GPL29263\r'] | rel= ['SuperSeries of: GSE159785', 'SuperSeries of: GSE159787']
 | n= 51 | plat= ['GPL1790\r'] | rel= []
 | n= 25 | plat= ['GPL96\r'] | rel= []
done


In [27]:
import urllib.request
exec("def H(g,p):\n    s=g[:-3]+'nnn'\n    u='https://ftp.ncbi.nlm.nih.gov/geo/series/%s/%s/matrix/%s-%s_series_matrix.txt.gz'%(s,g,g,p)\n    try:\n        r=urllib.request.urlopen(u,timeout=90)\n        print('OK  ',g,p,r.headers.get('Content-Length'),'bytes')\n    except Exception as e:\n        print('FAIL',g,p,str(e)[:60])\n")
T=[("GSE2283","GPL1874"),("GSE2283","GPL1875"),("GSE104948","GPL22945"),("GSE104948","GPL24120"),("GSE16161","GPL570"),("GSE16161","GPL571"),("GSE162911","GPL21697")]
z=[H(g,p) for g,p in T]
print('done')

OK   GSE2283 GPL1874 67531 bytes
OK   GSE2283 GPL1875 1906181 bytes
OK   GSE104948 GPL22945 4730443 bytes
OK   GSE104948 GPL24120 8237008 bytes
OK   GSE16161 GPL570 5476735 bytes
OK   GSE16161 GPL571 1931920 bytes
OK   GSE162911 GPL21697 20934 bytes
done


In [28]:
import urllib.request,gzip,re,collections
u="https://ftp.ncbi.nlm.nih.gov/geo/datasets/GDS1nnn/GDS1597/soft/GDS1597_full.soft.gz"
t=gzip.decompress(urllib.request.urlopen(u,timeout=120).read()).decode("utf-8","replace")
hdr=t.split("!dataset_table_begin")[0]
print("SOFT header chars:",len(hdr))
z=[print(l) for l in hdr.splitlines() if l.startswith("!subset") or l.startswith("!dataset_description") or l.startswith("!dataset_title") or l.startswith("!dataset_sample_count") or l.startswith("!dataset_value_type")]
print("done")

SOFT header chars: 10332
!dataset_title = Coronary artery atherosclerosis
!dataset_description = Analysis of 51 coronary artery segments from the explanted hearts of 22 cardiac patients. Results provide insight into the pathogenesis and progression of atherosclerotic disease.
!dataset_sample_count = 51
!dataset_value_type = log2 ratio
!subset_dataset_id = GDS1597
!subset_description = patient 1
!subset_sample_id = GSM38712
!subset_type = individual
!subset_dataset_id = GDS1597
!subset_description = patient 2
!subset_sample_id = GSM38713,GSM38714,GSM38715,GSM38716
!subset_type = individual
!subset_dataset_id = GDS1597
!subset_description = patient 3
!subset_sample_id = GSM38717,GSM38718,GSM38719,GSM38720,GSM38721
!subset_type = individual
!subset_dataset_id = GDS1597
!subset_description = patient 4
!subset_sample_id = GSM38722,GSM38723,GSM38724
!subset_type = individual
!subset_dataset_id = GDS1597
!subset_description = patient 5
!subset_sample_id = GSM38725,GSM38726,GSM38727
!subset_ty

In [29]:
import re,collections
B=[b for b in hdr.split("!subset_dataset_id") if "subset_type" in b]
g=lambda b,k: (re.search(k+r" = (.+)",b).group(1).strip() if re.search(k+r" = (.+)",b) else "")
rows=[(g(b,"!subset_type"),g(b,"!subset_description"),len(g(b,"!subset_sample_id").split(","))) for b in B]
print("subset types:",collections.Counter(t for t,_,_ in rows))
print()
z=[print(f"  {t:14s} {d:34s} n={n}") for t,d,n in rows if t!="individual"]
print()
print("TITLE:",[l for l in hdr.splitlines() if l.startswith("!dataset_title")])
print("VALUE:",[l for l in hdr.splitlines() if l.startswith("!dataset_value_type")])

subset types: Counter({'individual': 22})


TITLE: ['!dataset_title = Coronary artery atherosclerosis']
VALUE: ['!dataset_value_type = log2 ratio']


In [30]:
import urllib.request,gzip,os
exec("def dl(g,p):\n    f=g+'-'+p+'_series_matrix.txt'\n    if os.path.exists(f):\n        print('cached',f); return f\n    s=g[:-3]+'nnn'\n    u='https://ftp.ncbi.nlm.nih.gov/geo/series/%s/%s/matrix/%s-%s_series_matrix.txt.gz'%(s,g,g,p)\n    t=gzip.decompress(urllib.request.urlopen(u,timeout=240).read()).decode('utf-8','replace')\n    open(f,'w',encoding='utf-8').write(t)\n    print('wrote',f,len(t)//1000,'KB')\n    return f\n")
PP=[("GSE2283","GPL1874"),("GSE2283","GPL1875"),("GSE104948","GPL22945"),("GSE104948","GPL24120"),("GSE16161","GPL570"),("GSE16161","GPL571")]
F=[dl(g,p) for g,p in PP]
print(len(F),"files")

wrote GSE2283-GPL1874_series_matrix.txt 288 KB
wrote GSE2283-GPL1875_series_matrix.txt 5769 KB
wrote GSE104948-GPL22945_series_matrix.txt 10410 KB
wrote GSE104948-GPL24120_series_matrix.txt 18229 KB
wrote GSE16161-GPL570_series_matrix.txt 12420 KB
wrote GSE16161-GPL571_series_matrix.txt 4278 KB
6 files


In [31]:
ns=lambda f: max([len(l.split(chr(9)))-1 for l in rd(f) if l.lower().startswith("!sample_geo_accession")]+[0])
z=[print(f.replace("_series_matrix.txt","")," n=",ns(f)) for f in F]
print()
c2=lambda f,tag,k: [print(f.replace("_series_matrix.txt","")[:20],tag,i,c) for i,s in enumerate(pick(rd(f),k)) for c in [vc(s)] if 2<=len(set(v for v in s.split(chr(9))))<=8]
z=[c2(f,"T","!sample_title") for f in F]
z=[c2(f,"S","!sample_source_name") for f in F]
z=[c2(f,"C","!sample_characteristics") for f in F]
print("done")

GSE2283-GPL1874  n= 50
GSE2283-GPL1875  n= 136
GSE104948-GPL22945  n= 71
GSE104948-GPL24120  n= 125
GSE16161-GPL570  n= 18
GSE16161-GPL571  n= 15

GSE104948-GPL22945 C 1 [('diagnosis: ANCA Associated Vasculitis', 22), ('', 18), ('diagnosis: Focal Segmental Glomerular Sclerosis', 10), ('diagnosis: Diabetic Nephropathy', 7), ('diagnosis: Focal Segmental Glomerular Sclerosis/Minima', 6)]
GSE16161-GPL570 C 2 [('condition: atopic dermatitis (AD)', 9), ('condition: Normal', 9)]
done


In [32]:
Q=["GSE2283-GPL1874_series_matrix.txt","GSE2283-GPL1875_series_matrix.txt","GSE104948-GPL24120_series_matrix.txt","GSE16161-GPL571_series_matrix.txt"]
al=lambda f,tag,k: [print(f[:18],tag,i,len(set(s.split(chr(9)))),vc(s)) for i,s in enumerate(pick(rd(f),k))]
z=[al(f,"T","!sample_title") for f in Q]
print()
z=[al(f,"S","!sample_source_name") for f in Q]
print()
z=[al(f,"C","!sample_characteristics") for f in Q]
print("done")

GSE2283-GPL1874_se T 0 50 [('NB2211', 1), ('NB2212', 1), ('NB2213', 1), ('NB2214', 1), ('NB2215', 1)]
GSE2283-GPL1875_se T 0 136 [('S001', 1), ('S002', 1), ('S003', 1), ('S004', 1), ('S005', 1)]
GSE104948-GPL24120 T 0 125 [('H1-Glom-DN1', 1), ('H1-Glom-DN10', 1), ('H1-Glom-DN2', 1), ('H1-Glom-DN5', 1), ('H1-Glom-DN8', 1)]
GSE16161-GPL571_se T 0 15 [('PS Chip#1', 1), ('PS Chip#2', 1), ('PS Chip#3', 1), ('PS Chip#4', 1), ('PS Chip#5', 1)]

GSE2283-GPL1874_se S 0 1 [('Mixture of human neuroblastoma cell lines reference RN', 50)]
GSE2283-GPL1874_se S 1 50 [('NB2211', 1), ('NB2212', 1), ('NB2213', 1), ('NB2214', 1), ('NB2215', 1)]
GSE2283-GPL1875_se S 0 1 [('Mixture of human neuroblastoma cell lines reference RN', 136)]
GSE2283-GPL1875_se S 1 136 [('S001', 1), ('S002', 1), ('S003', 1), ('S004', 1), ('S005', 1)]
GSE104948-GPL24120 S 0 125 [('Kidney Biopsy from Human H1-Glom-DN1', 1), ('Kidney Biopsy from Human H1-Glom-DN10', 1), ('Kidney Biopsy from Human H1-Glom-DN2', 1), ('Kidney Biopsy fr

In [33]:
import collections
full=lambda f,k,i: collections.Counter(v.strip().strip(chr(34)) for v in pick(rd(f),k)[i].split(chr(9)))
print("GSE104948-GPL24120 diagnosis (C line 1):")
z=[print("   ",n,"x",v[:70]) for v,n in full("GSE104948-GPL24120_series_matrix.txt","!sample_characteristics",1).most_common()]
print()
print("GSE16161-GPL570 condition (C line 2):")
z=[print("   ",n,"x",v[:70]) for v,n in full("GSE16161-GPL570_series_matrix.txt","!sample_characteristics",2).most_common()]
print()
print("GSE104948-GPL22945 diagnosis (C line 1):")
z=[print("   ",n,"x",v[:70]) for v,n in full("GSE104948-GPL22945_series_matrix.txt","!sample_characteristics",1).most_common()]
print("done")

GSE104948-GPL24120 diagnosis (C line 1):
    32 x diagnosis: Systemic Lupus Erythematosus
    27 x diagnosis: IgA Nephropathy
    21 x diagnosis: Membranous Glomerulonephropathy
    15 x diagnosis: Hypertensive Nephropathy
    9 x diagnosis: Minimal Change Disease
    8 x diagnosis: Focal Segmental Glomerular Sclerosis
    5 x diagnosis: Diabetic Nephropathy
    3 x diagnosis: Thin Membrande Disease
    3 x 
    2 x diagnosis: Tumor Nephrectomy

GSE16161-GPL570 condition (C line 2):
    9 x condition: atopic dermatitis (AD)
    9 x condition: Normal

GSE104948-GPL22945 diagnosis (C line 1):
    22 x diagnosis: ANCA Associated Vasculitis
    18 x 
    10 x diagnosis: Focal Segmental Glomerular Sclerosis
    7 x diagnosis: Diabetic Nephropathy
    6 x diagnosis: Focal Segmental Glomerular Sclerosis/Minimal Change Disease
    5 x diagnosis: Minimal Change Disease
    3 x diagnosis: Tumor Nephrectomy
done


In [34]:
import urllib.request,gzip,os
f="GSE3960_series_matrix.txt"
u="https://ftp.ncbi.nlm.nih.gov/geo/series/GSE3nnn/GSE3960/matrix/GSE3960_series_matrix.txt.gz"
open(f,"w",encoding="utf-8").write(gzip.decompress(urllib.request.urlopen(u,timeout=240).read()).decode("utf-8","replace")) if not os.path.exists(f) else print("cached")
print("GSE3960 n=",ns(f))
z=[al(f,"T","!sample_title") for f in [f]]
z=[al(f,"S","!sample_source_name") for f in [f]]
z=[al(f,"C","!sample_characteristics") for f in [f]]
print("done")

GSE3960 n= 102
GSE3960_series_mat T 0 102 [('Fetal Brain (HG-U95A)', 1), ('37 (HG-U95A)', 1), ('401 (HG-U95A)', 1), ('406 (HG-U95A)', 1), ('491 (HG-U95A)', 1)]
GSE3960_series_mat S 0 2 [('Primary Tumor', 101), ('Fetal Brain', 1)]
GSE3960_series_mat C 0 102 [('Control', 1), ('Age Diagnosis: 38, INSS Stage:1, MYCN:Not Amplified, 1', 1), ('Age Diagnosis: 392, INSS Stage:1, MYCN:Not Amplified, ', 1), ('Age Diagnosis: 141, INSS Stage:1, MYCN:Not Amplified, ', 1), ('Age Diagnosis: 69, INSS Stage:1, MYCN:Not Amplified, 1', 1)]
done


In [2]:
import urllib.request,gzip,os,re,collections
exec("def dl(g):\n    f=g+'_series_matrix.txt'\n    if os.path.exists(f):\n        return f\n    s=g[:-3]+'nnn'\n    u='https://ftp.ncbi.nlm.nih.gov/geo/series/%s/%s/matrix/%s_series_matrix.txt.gz'%(s,g,g)\n    open(f,'w',encoding='utf-8').write(gzip.decompress(urllib.request.urlopen(u,timeout=300).read()).decode('utf-8','replace'))\n    print('downloaded',f)\n    return f\n")
rd=lambda p: open(p,encoding="utf-8",errors="replace").read().split("!series_matrix_table_begin")[0].splitlines()
pick=lambda ls,k: [l.split(chr(9),1)[1] for l in ls if l.lower().startswith(k)]
ns=lambda f: max([len(l.split(chr(9)))-1 for l in rd(f) if l.lower().startswith("!sample_geo_accession")]+[0])
full=lambda f,k,i: collections.Counter(v.strip().strip(chr(34)) for v in pick(rd(f),k)[i].split(chr(9)))
nlines=lambda f,k: len(pick(rd(f),k))
print("helpers ready")

helpers ready


In [3]:
F12=dl("GSE35007"); F13=dl("GSE134662")
showall=lambda f,k: [print("   ",k[8:12].upper(),i,"->",full(f,k,i).most_common(8)) for i in range(nlines(f,k)) if 2<=len(full(f,k,i))<=8]
print("ROW 12  GSE35007  n =",ns(F12))
z=[showall(F12,x) for x in ("!sample_title","!sample_source_name","!sample_characteristics")]
print()
print("ROW 13  GSE134662  n =",ns(F13))
z=[showall(F13,x) for x in ("!sample_title","!sample_source_name","!sample_characteristics")]
import re
T13=[x.strip().strip(chr(34)) for x in pick(rd(F13),"!sample_title")[0].split(chr(9))]
print("   titles matching ^Normal:",sum(1 for x in T13 if re.match(r'^Normal',x)),"of",len(T13))
print("   sample of titles:",T13[:4],T13[-4:])

downloaded GSE35007_series_matrix.txt
downloaded GSE134662_series_matrix.txt
ROW 12  GSE35007  n = 311
    CHAR 1 -> [('clinical status: E', 133), ('clinical status: FU', 80), ('clinical status: C', 61), ('clinical status: A', 36), ('severity score: 0.281', 1)]
    CHAR 3 -> [('Sex: M', 159), ('Sex: F', 151), ('hb phenotype: SC', 1)]
    CHAR 4 -> [('hb phenotype: SS', 190), ('hb phenotype: SC', 59), ('hb phenotype: AS', 30), ('hb phenotype: A', 19), ('hb phenotype: AA', 12), ('white blood cells: 13', 1)]

ROW 13  GSE134662  n = 54
    SOUR 0 -> [('GVHD Colon Biopsie_onset', 22), ('GVHD Colon Biopsie_Refractory', 22), ('Healthy Colon Biopsie', 10)]
    CHAR 0 -> [('gvhd stage: Onset', 22), ('gvhd stage: Refractory', 22), ('gvhd stage: Control', 10)]
    CHAR 2 -> [('gender: Male', 32), ('gender: Female', 12), ('', 10)]
   titles matching ^Normal: 10 of 54
   sample of titles: ['Normal1', 'Normal10', 'Normal2', 'Normal3'] ['Sample5.G2', 'Sample6.G2', 'Sample7.G2', 'Sample9.G2']


In [4]:
print("R12 hb phenotype (C4):")
z=[print("  ",n,v[:48]) for v,n in full(F12,"!sample_characteristics",4).most_common()]
print("R12 clinical status (C1):")
z=[print("  ",n,v[:48]) for v,n in full(F12,"!sample_characteristics",1).most_common()]
print("R13 gvhd stage (C0):")
z=[print("  ",n,v[:48]) for v,n in full(F13,"!sample_characteristics",0).most_common()]

R12 hb phenotype (C4):
   190 hb phenotype: SS
   59 hb phenotype: SC
   30 hb phenotype: AS
   19 hb phenotype: A
   12 hb phenotype: AA
   1 white blood cells: 13
R12 clinical status (C1):
   133 clinical status: E
   80 clinical status: FU
   61 clinical status: C
   36 clinical status: A
   1 severity score: 0.281
R13 gvhd stage (C0):
   22 gvhd stage: Onset
   22 gvhd stage: Refractory
   10 gvhd stage: Control


In [ ]:
Q0="H4sIAFqLvmoC/7U723LbyHLv+ooxvFUEbRISLVvycleboiXYVh1ZUkh6vS5ZYYHEkMQhCOBgAMo8DP8iT8lLvu18Sbp7ZoABLyv5JNHWWuRcenr63j2t588Oc5EeDoPokEcLliyzaRwdH1iWNQo9IYLxcpDGIuOpcJIlazbZMA9Cn2VTzkae4IejOMrSOGRyERvHKePeaMqGPBpN5146Y3/Lebp0Dg66ecgF81LOfA6wU+6zj27XbTBvGC84QfS9zGuwcRrP6es44HBUEC14lMXpEj4ddLuDf/3sdr8Oep+7v7tfmRcpXOJoHKRzLwviSMBCBgu7N196bn/QdXs3V7+7Fw7rT/mSEEjSIMq4f+CFcTQRgc8ZXwCSLOUiDzPA9R//9R+s/9Fl3c9XLnzo9FnvY+fW7cHnyx57f3nltpnHFl4aeMOQswDu5fuwW8ClhoBoJmCRe3XBrjuf3MZBhOD1xGjqpd4IiBWILBixq8trl11eX7h/OOyDe8Me0iADOsURZ8963jwJ+aC6QxyEAUwmAHEDVO/qpt9gUZzRpMauwUQMyAqCBUxL02UQTWDED8ZjngJtD6J8PoQd8XgDomBiGowBZ0me0MsMuCxPGHDbjx8iAIY4OewT90QOnD0ADnzoucdvjo5O2zTHXgN1PV+w2nTIkikHLJcJr5HAHLeOFHqCGFp7mAIN2DCMY5+NeBiK2gGui/GIy8jn3/ECQM4kFgFynIkghHuESzYPROgNeShAKmLBNVhgKYMfKd1st2QLHo6B8hkufA5UZBHPHuJ09tjO6s9zKVxCKYRoaIbChaeA9uN4KFFW0NRulHGlYue93wXq50EwT+I0A9"
Q1="EPQz4iwW+wkVg02OTvQdJgMXxNkftL+JCnYRgMnZSDNors4OA5HPS//ZFIoVYDuIcpiFKbWShAFrM9FnlzUIZNeSKVrjeYlQVZyK0GSpAl4jwdccsBMLcpH3EfbAcplfvH+dXnCzARcFLEzm+u+92bK/2t09MTeYR0nETcd9i5tEcAKs6z1ItmguwU2KORl8PvWhRHzbEXgExOamg2Mi8Ai1HTQ84B6nyPnbHVAbNAhlunr09en1ptUJdRZqfxw5l1AnjjD1m2M2suBqNIqDH4IVKcSUo05I3PLD8QHPEQGSiR1Thg2z+jLA3P7lLr35RNte4bhDuNzcVP+J1/H4W5D0P39YbE79WbkzfHP1fwe1vFL/GmgzCPJhpDhZ+iusbwGrTrx9EKAHgVr50wQPg50mASZF7IRAJ22gt/YYnguR8P83DGsphM3sILc2nblL16576/6bpgFpbErNQTmaWv3jo6Pn4NNzeu3jqSt1RXnwBLg4VX8GYna9SikGWBEHi6tk6PsGnKvTCbLiv0UGz2d/OK7OEGwq8qCJPNK0VpN8KmBX0Ex85PsA1/d36q4NnrVcUJh871Wprbz8beOTm80n+wiULmF9bp4Vy/27kET+SRtwmQmSj1DfTLEQ9AZ1Nww/OSj8evT05eVUS4dVzl42LqD8DQxZEc3s1HWIQnTZ6oX1WKgPnkmaJAysdotSDm2KN0b06Pj99uMPL1YesNbFcYT8eD+TKG+/tBPofhR41CLv4ca7Bbzr8oK2UiTkfEcBYoFQgj4W8se5Jidq66bufiK3v3+fKqj1zSUd2rXxjizTzpgW5vepf9y99dbYrJe4NbgzhhlAZJqZkn"
Q2="p6CIQB6TPqcVjiYihjBCBGIgZkFUEkj7hScapW/DPIIAMQ4X3P82NOnyDUIUAU7RC+X4kwhBsQNKMNAAL9y/7F+5v2x5MQqi2PUN0wyUkaqXMS8MS7E+gf+aH26v3pweVYX7bSknnq8JsE+sgRc+RTmPSHUUQ/xbsc5eFicQFvqcAmOwblvCvD5w/7h1z/vo7jDaoNB7O3QGkFM+moE/9yboLjPl+iOHdZAG3iTlfI6WQBoGPoaAxNnwn3br7asGQ46i/zc9l916U53Rht0m/quZqgm1Wz8fNSBy1Du0CbFfv26w1pEa1npqt05PYfj4ZENA7dYxDmsgVYbZPzfYz0SkA7gRG0RDu96GkCrL04hZQbKc8TTioYXCAkGWM499SnEwpJnE8STkDpgsb7i5QIKDiGwA6mN7o1G9TYydABfgmyOSMMhsq2nV747uaUZk+RAmJ3ft5vE9e8msKIoA6phB1GtP6uw3dsIg5uVMhpcKxbE1zbJEtA8Px1niRKNh4ETh3ImCqTOJF4cTHh8KEGkuDld4wPpwNVkfgqSkwffDFSCyHsjpgRxzsu+ZM/m7JdEf82w0ReRR3EA6zizHUvdIANUYglmwR85f4yCyaQHIs7UHqlWnfXAfvQ0ifJEJO1EQjUvFCVwZolsIEWMfbNyZlWfjJioUT9M4FWdWypPQg8Cm7mDCYUvYD+B1NoJgB74SNJMTEFEGcw6h49nx0VEd7V5aopAhDyC2diCBjecJ5nt2qk6p06DP7f34lIjoS1gP1vZN6NRxeerYoRzAziQASi/sscUo8wpjzwe1XCVrZq9QGrL6YYuftJ3WeM0+vaurUxXxMsm7xEsFwO"


In [10]:
R1="aW1wb3J0IG9zLHRpbWUsbnVtcHkgYXMgbnAscGFuZGFzIGFzIHBkLGZzc3BlYyxoNXB5CnQwPXRpbWUudGltZSgpCmxtPXNldChpbnQoeCkgZm9yIHggaW4gbnAubG9hZCgibGFuZG1hcmtfb3ZlcmxhcC5ucHkiKSkKcHJpbnQoImxhbmRtYXJrcyB0byBrZWVwOiIsbGVuKGxtKSkKcz1wZC5yZWFkX2Nzdigic2lnaW5mb19iZXRhLnR4dCIsc2VwPSJcdCIsdXNlY29scz1bInNpZ19pZCIsInBlcnRfaWQiLCJwZXJ0X3R5cGUiLCJpc19leGVtcGxhcl9zaWciLCJxY19wYXNzIl0sbG93X21lbW9yeT1GYWxzZSkKcz1zWyhzLnBlcnRfdHlwZT09InRydF9jcCIpJihzLmlzX2V4ZW1wbGFyX3NpZz09MSkmKHMucWNfcGFzcz09MSldCnByaW50KCJxYyBleGVtcGxhciB0cnRfY3Agc2lnbmF0dXJlczoiLGxlbihzKSwiY29tcG91bmRzOiIscy5wZXJ0X2lkLm51bmlxdWUoKSkKdT0iaHR0cHM6Ly9zMy5hbWF6b25hd3MuY29tL21hY2NoaWF0by5jbHVlLmlvL2J1aWxkcy9MSU5D"
R2="UzIwMjAvbGV2ZWw1L2xldmVsNV9iZXRhX3RydF9jcF9uNzIwMjE2eDEyMzI4LmdjdHgiCmg9aDVweS5GaWxlKGZzc3BlYy5vcGVuKHUsInJiIikub3BlbigpLCJyIik7IGQ9aFsiMCJdWyJEQVRBIl1bIjAiXVsibWF0cml4Il0KY29saWQ9bnAuYXJyYXkoW3guZGVjb2RlKCkgZm9yIHggaW4gaFsiMCJdWyJNRVRBIl1bIkNPTCJdWyJpZCJdWzpdXSkKcm93aWQ9bnAuYXJyYXkoW2ludCh4LmRlY29kZSgpKSBmb3IgeCBpbiBoWyIwIl1bIk1FVEEiXVsiUk9XIl1bImlkIl1bOl1dKQpwcmludCgibWF0cml4IixkLnNoYXBlLCJDT0woc2lnKSBpZHMiLGxlbihjb2xpZCksIlJPVyhnZW5lKSBpZHMiLGxlbihyb3dpZCkpCmFzc2VydCBkLnNoYXBlWzBdPT1sZW4oY29saWQpIGFuZCBkLnNoYXBlWzFdPT1sZW4ocm93aWQpLCAiQVhJUyBMQUJFTFMgTk9UIEFTIEVYUEVDVEVEIgpnY29sPW5wLmFycmF5KFtpIGZvciBpLGcgaW4gZW51bWVyYXRlKHJvd2lkKSBpZiBnIGluIGxt"
R3="XSkKcHJpbnQoImxhbmRtYXJrIGNvbHVtbiBpbmRpY2VzOiIsbGVuKGdjb2wpKQpwb3M9e3NpZDppIGZvciBpLHNpZCBpbiBlbnVtZXJhdGUoY29saWQpfQp3YW50PVsocG9zW3hdLHApIGZvciB4LHAgaW4gemlwKHMuc2lnX2lkLHMucGVydF9pZCkgaWYgeCBpbiBwb3NdCnByaW50KCJzaWduYXR1cmVzIGxvY2F0YWJsZSBpbiBtYXRyaXg6IixsZW4od2FudCksIm9mIixsZW4ocykpCndhbnQuc29ydCgpCmlkeD1ucC5hcnJheShbd1swXSBmb3IgdyBpbiB3YW50XSk7IHBlcnRzPW5wLmFycmF5KFt3WzFdIGZvciB3IGluIHdhbnRdKQpvdXQ9bnAuZW1wdHkoKGxlbihpZHgpLGxlbihnY29sKSksZHR5cGU9bnAuZmxvYXQzMikKQj0yMDAwOyB0MT10aW1lLnRpbWUoKQpmb3IgYSBpbiByYW5nZSgwLGxlbihpZHgpLEIpOgogICAgYj1taW4oYStCLGxlbihpZHgpKQogICAgb3V0W2E6Yl09ZFtpZHhbYTpiXSw6XVs6LGdjb2xdCiAgICBpZiAoYS8vQiklNT09MDoKICAgICAg"
R4="ICBlbD10aW1lLnRpbWUoKS10MTsgZG9uZT1iL2xlbihpZHgpCiAgICAgICAgcHJpbnQoIiAgJTZkLyU2ZCAgJTQuMWYlJSAgJTUuMGZzIGVsYXBzZWQgIGV0YSAlNS4wZnMiJShiLGxlbihpZHgpLDEwMCpkb25lLGVsLGVsL21heChkb25lLDFlLTkpLWVsKSxmbHVzaD1UcnVlKQpwcmludCgicmVhZCBkb25lIGluIixyb3VuZCh0aW1lLnRpbWUoKS10MSwxKSwicyIpCm5wLnNhdmUoImxtX21hdHJpeC5ucHkiLG91dCk7IG5wLnNhdmUoImxtX3BlcnRzLm5weSIscGVydHMpOyBucC5zYXZlKCJsbV9nZW5lcy5ucHkiLHJvd2lkW2djb2xdKQpwcmludCgic2F2ZWQgbG1fbWF0cml4Lm5weSIsb3V0LnNoYXBlLG91dC5kdHlwZSxyb3VuZChvdXQubmJ5dGVzLzFlNiwxKSwiTUIiKQpwcmludCgiZmluaXRlIGZyYWN0aW9uIixyb3VuZChmbG9hdChucC5pc2Zpbml0ZShvdXQpLm1lYW4oKSksNikpCnByaW50KCJUT1RBTCIscm91bmQodGltZS50aW1lKCktdDAsMSksInMiKQo="

In [11]:
import base64,hashlib
B=R1+R2+R3+R4
print("len",len(B),"want",2624)
print("md5",hashlib.md5(B.encode()).hexdigest(),"want","534c6d1e728a3757d2838c9623a0bbbf")
assert len(B)==2624 and hashlib.md5(B.encode()).hexdigest()=="534c6d1e728a3757d2838c9623a0bbbf","TRANSFER CORRUPTED"
exec(base64.b64decode(B))

len 2624 want 2624
md5 534c6d1e728a3757d2838c9623a0bbbf want 534c6d1e728a3757d2838c9623a0bbbf
landmarks to keep: 951
qc exemplar trt_cp signatures: 90834 compounds: 26425
matrix (720216, 12328) COL(sig) ids 720216 ROW(gene) ids 12328
landmark column indices: 951
signatures locatable in matrix: 90834 of 90834
    2000/ 90834   2.2%     46s elapsed  eta  2034s
   12000/ 90834  13.2%    109s elapsed  eta   717s
   22000/ 90834  24.2%    163s elapsed  eta   511s
   32000/ 90834  35.2%    221s elapsed  eta   407s
   42000/ 90834  46.2%    294s elapsed  eta   342s
   52000/ 90834  57.2%    364s elapsed  eta   272s
   62000/ 90834  68.3%    564s elapsed  eta   262s
   72000/ 90834  79.3%    605s elapsed  eta   158s
   82000/ 90834  90.3%    739s elapsed  eta    80s
   90834/ 90834  100.0%   1038s elapsed  eta     0s
read done in 1037.5 s
saved lm_matrix.npy (90834, 951) float32 345.5 MB
finite fraction 1.0
TOTAL 1056.6 s


In [12]:
S1="aW1wb3J0IGNzdixnemlwLG9zLHRpbWUsbnVtcHkgYXMgbnAscGFuZGFzIGFzIHBkCnQwPXRpbWUudGltZSgpCkdTRT0iR1NFNTczMzgiOyBLPTE1MDsgTlM9MjA7IFNFRUQ9MTIzNDUKTT1ucC5sb2FkKCJsbV9tYXRyaXgubnB5Iik7IHBlcnRzPW5wLmxvYWQoImxtX3BlcnRzLm5weSIsYWxsb3dfcGlja2xlPVRydWUpOyBnZW5lcz1ucC5sb2FkKCJsbV9nZW5lcy5ucHkiKQpwcmludCgibGlicmFyeSIsTS5zaGFwZSwiY29tcG91bmRzIixsZW4oc2V0KHBlcnRzLnRvbGlzdCgpKSkpCnJvcz17clsiZ3NtIl06clsiZ3JvdXAiXSBmb3IgciBpbiBjc3YuRGljdFJlYWRlcihvcGVuKEdTRSsiX3Jvc3Rlci5jc3YiLGVuY29kaW5nPSJ1dGYtOCIpKX0KIyBwcm9iZSAtPiBlbnRyZXosIHJlY29tcHV0ZWQgZnJvbSB0aGUgY2FjaGVkIGFubm90YXRpb24gcmF0aGVyIHRoYW4ga2VybmVsIHN0YXRlCnBsYXQ9IkdQTDExNTMyIjsgcDJlPXt9OyBpbmI9RmFsc2U7IGNvbHM9Tm9uZTsgaWdlPU5vbmUKZm9yIHJhdyBpbiBnemlwLm9wZW4ocGxhdCsiLmFubm90Lmd6IiwicnQiLGVuY29kaW5nPSJ1dGYtOCIsZXJyb3JzPSJyZXBsYWNlIik6CiAgICBpZiByYXcuc3RhcnRzd2l0aCgiIXBsYXRmb3JtX3RhYmxlX2VuZCIpOiBicmVhawogICAgaWYgaW5iOgogICAgICAgIHA9cmF3LnJzdHJpcCgiXG4iKS5zcGxpdCgiXHQiKQogICAgICAgIGlmIGNvbHMgaXMgTm9uZTogY29scz1wOyBpZ2U9W2kgZm9yIGksYyBpbiBlbnVtZXJhdGUoY29scykgaWYgIkdlbmUgSUQiIGluIGNdCiAgICAgICAgZWxpZiBpZ2UgYW5kIGxlbihwKT5pZ2VbMF06CiAgICAgICAgICAgIGc9cFtp"
S2="Z2VbMF1dLnNwbGl0KCIvLy8iKVswXS5zdHJpcCgpCiAgICAgICAgICAgIGlmIGcuaXNkaWdpdCgpOiBwMmVbcFswXV09aW50KGcpCiAgICBpZiByYXcuc3RhcnRzd2l0aCgiIXBsYXRmb3JtX3RhYmxlX2JlZ2luIik6IGluYj1UcnVlCmdpPXtnOmkgZm9yIGksZyBpbiBlbnVtZXJhdGUoZ2VuZXMpfQpoZHI9Tm9uZTsgYWNjPU5vbmU7IHN1bXM9e30KaW5iPUZhbHNlCmZvciBsaW5lIGluIG9wZW4oR1NFKyJfc2VyaWVzX21hdHJpeC50eHQiLGVuY29kaW5nPSJ1dGYtOCIsZXJyb3JzPSJyZXBsYWNlIik6CiAgICBpZiBsaW5lLnN0YXJ0c3dpdGgoIiFzZXJpZXNfbWF0cml4X3RhYmxlX2VuZCIpOiBicmVhawogICAgaWYgaW5iOgogICAgICAgIHA9bGluZS5yc3RyaXAoIlxuIikuc3BsaXQoIlx0IikKICAgICAgICBpZiBoZHIgaXMgTm9uZTogaGRyPVt4LnN0cmlwKCkuc3RyaXAoY2hyKDM0KSkgZm9yIHggaW4gcF07IGNvbnRpbnVlCiAgICAgICAgcGlkPXBbMF0uc3RyaXAoKS5zdHJpcChjaHIoMzQpKTsgZT1wMmUuZ2V0KHBpZCkKICAgICAgICBpZiBlIGlzIE5vbmUgb3IgZSBub3QgaW4gZ2k6IGNvbnRpbnVlCiAgICAgICAgdHJ5OiB2YWxzPW5wLmFycmF5KFtmbG9hdCh4KSBmb3IgeCBpbiBwWzE6XV0sZHR5cGU9bnAuZmxvYXQ2NCkKICAgICAgICBleGNlcHQgVmFsdWVFcnJvcjogY29udGludWUKICAgICAgICBzdW1zLnNldGRlZmF1bHQoZ2lbZV0sW10pLmFwcGVuZCh2YWxzKQogICAgaWYgbGluZS5zdGFydHN3aXRoKCIhc2VyaWVzX21hdHJpeF90YWJsZV9iZWdpbiIpOiBpbmI9VHJ1ZQpzYW1wbGVzPWhkclsxOl0KcHJpbnQoInNhbXBsZXMgaW4g"
S3="dGFibGUiLGxlbihzYW1wbGVzKSwiZ2VuZXMgd2l0aCA+PTEgcHJvYmUiLGxlbihzdW1zKSkKRT1ucC5mdWxsKChsZW4oZ2VuZXMpLGxlbihzYW1wbGVzKSksbnAubmFuKQpucHJvYmU9W10KZm9yIGcsbHN0IGluIHN1bXMuaXRlbXMoKToKICAgIEE9bnAudnN0YWNrKGxzdCk7IG5wcm9iZS5hcHBlbmQoQS5zaGFwZVswXSk7IEVbZ109bnAubWVkaWFuKEEsYXhpcz0wKSAgICMgUFJFLVJFR0lTVEVSRUQ6IG1lZGlhbiBhY3Jvc3MgcHJvYmVzCnByaW50KCJwcm9iZXMgcGVyIGdlbmU6IG1lZGlhbiIsaW50KG5wLm1lZGlhbihucHJvYmUpKSwibWF4IixpbnQobnAubWF4KG5wcm9iZSkpKQprZWVwPX5ucC5pc25hbihFKS5hbnkoYXhpcz0xKTsgRT1FW2tlZXBdOyBwcmludCgiZ2VuZXMgdXNhYmxlIixpbnQoa2VlcC5zdW0oKSksIm9mIixsZW4oZ2VuZXMpKQpNaz1NWzosa2VlcF0KY2FzZXM9bnAuYXJyYXkoW2kgZm9yIGkscyBpbiBlbnVtZXJhdGUoc2FtcGxlcykgaWYgcm9zLmdldChzKT09ImNhc2UiXSkKY3RybHM9bnAuYXJyYXkoW2kgZm9yIGkscyBpbiBlbnVtZXJhdGUoc2FtcGxlcykgaWYgcm9zLmdldChzKT09ImNvbnRyb2wiXSkKcHJpbnQoImNhc2VzIixsZW4oY2FzZXMpLCJjb250cm9scyIsbGVuKGN0cmxzKSkKZGVmIGtzKHJwLG4sdCk6CiAgICBpZiB0PT0wIG9yIG49PTA6IHJldHVybiAwLjAKICAgIFY9bnAuc29ydChucC5hc2FycmF5KHJwLGZsb2F0KSk7IGo9bnAuYXJhbmdlKDEsdCsxLGR0eXBlPWZsb2F0KQogICAgYT1ucC5tYXgoai90LVYvbik7IGI9bnAubWF4KFYvbi0oai0xLjApL3QpCiAgICByZXR1cm4gZmxvYXQoYSkgaWYgYT5i"
S4="IGVsc2UgZmxvYXQoLWIpCmRlZiBjb25uX2FsbChNeCx1cCxkbik6CiAgICBuPU14LnNoYXBlWzFdOyBvdXQ9bnAuZW1wdHkoTXguc2hhcGVbMF0pCiAgICBvcmRlcj1ucC5hcmdzb3J0KC1NeCxheGlzPTEsa2luZD0ic3RhYmxlIikKICAgIHJhbms9bnAuZW1wdHlfbGlrZShvcmRlcik7IGFyPW5wLmFyYW5nZSgxLG4rMSkKICAgIG5wLnB1dF9hbG9uZ19heGlzKHJhbmssb3JkZXIsbnAuYnJvYWRjYXN0X3RvKGFyLG9yZGVyLnNoYXBlKSxheGlzPTEpCiAgICBmb3IgaSBpbiByYW5nZShNeC5zaGFwZVswXSk6CiAgICAgICAgZXU9a3MocmFua1tpLHVwXSxuLGxlbih1cCkpOyBlZD1rcyhyYW5rW2ksZG5dLG4sbGVuKGRuKSkKICAgICAgICBvdXRbaV09MC4wIGlmIGV1KmVkPj0wIGVsc2UgKGV1LWVkKS8yLjAKICAgIHJldHVybiBvdXQKZGVmIHF1ZXJ5KGEsYixrKToKICAgIEE9RVs6LGFdOyBCPUVbOixiXQogICAgZD1BLm1lYW4oMSktQi5tZWFuKDEpCiAgICBwb29sZWQ9bnAuc3FydCgwLjUqKEEudmFyKDEsZGRvZj0xKStCLnZhcigxLGRkb2Y9MSkpKQogICAgZmw9bnAubWVkaWFuKHBvb2xlZFtwb29sZWQ+MF0pIGlmIG5wLmFueShwb29sZWQ+MCkgZWxzZSAxLjAKICAgIHN0PWQvbnAubWF4aW11bShwb29sZWQsZmwqMC4xKTsgbz1ucC5hcmdzb3J0KC1zdCxraW5kPSJzdGFibGUiKQogICAgcmV0dXJuIG9bOmtdLmNvcHkoKSxvWy1rOl0uY29weSgpCmRlZiBieWNvbXBvdW5kKHNjKToKICAgIGRmPXBkLkRhdGFGcmFtZSh7InAiOnBlcnRzLCJzIjpzY30pOyByZXR1cm4gZGYuZ3JvdXBieSgicCIpWyJzIl0ubWVkaWFuKCkgICMgUFJFLVJFR0lTVEVS"
S5="RUQ6IG1lZGlhbiBhY3Jvc3Mgc2lnbmF0dXJlcwpkZWYgbWlkcmFuayh2KToKICAgIHY9bnAuYXNhcnJheSh2LGZsb2F0KTsgbj12LnNpemU7IG89bnAuYXJnc29ydCh2LGtpbmQ9InN0YWJsZSIpOyByPW5wLmVtcHR5KG4pOyBpPTAKICAgIHdoaWxlIGk8bjoKICAgICAgICBqPWkKICAgICAgICB3aGlsZSBqKzE8biBhbmQgdltvW2orMV1dPT12W29baV1dOiBqKz0xCiAgICAgICAgcltvW2k6aisxXV09MC41KihpK2opKzEuMDsgaT1qKzEKICAgIHJldHVybiByCmRlZiBzcGVhcihhLGIpOgogICAgcmE9bWlkcmFuayhhKS1taWRyYW5rKGEpLm1lYW4oKTsgcmI9bWlkcmFuayhiKS1taWRyYW5rKGIpLm1lYW4oKQogICAgZGVuPW5wLnNxcnQoKHJhKnJhKS5zdW0oKSoocmIqcmIpLnN1bSgpKQogICAgcmV0dXJuIGZsb2F0KCJuYW4iKSBpZiBkZW49PTAgZWxzZSBmbG9hdCgocmEqcmIpLnN1bSgpL2RlbikKdXAsZG49cXVlcnkoY2FzZXMsY3RybHMsSykKcmVhbD1ieWNvbXBvdW5kKGNvbm5fYWxsKE1rLHVwLGRuKSkKcHJpbnQoIlJFQUwgcXVlcnk6IGNvbXBvdW5kcyIsbGVuKHJlYWwpLCJmcmFjIGV4YWN0bHkgemVybyAlLjRmIiVmbG9hdCgocmVhbD09MCkubWVhbigpKSwKICAgICAgInNwcmVhZCAlLjRmIiVmbG9hdChyZWFsLm1heCgpLXJlYWwubWluKCkpLGZsdXNoPVRydWUpCnJob3M9W10KZm9yIHMgaW4gcmFuZ2UoTlMpOgogICAgcm5nPW5wLnJhbmRvbS5kZWZhdWx0X3JuZyhTRUVEK3MpOyBwPXJuZy5wZXJtdXRhdGlvbihjdHJscyk7IGg9bGVuKGN0cmxzKS8vMgogICAgdTIsZDI9cXVlcnkocFs6aF0scFtoOjIqaF0sSykKICAgIG51bD1ieWNv"
S6="bXBvdW5kKGNvbm5fYWxsKE1rLHUyLGQyKSkKICAgIHI9c3BlYXIocmVhbC52YWx1ZXMsbnVsLnJlaW5kZXgocmVhbC5pbmRleCkudmFsdWVzKTsgcmhvcy5hcHBlbmQocikKICAgIHByaW50KCIgIHNwbGl0ICUyZCByaG8gJXMiJShzKzEsKCJuYW4iIGlmIHIhPXIgZWxzZSAiJSsuNGYiJXIpKSxmbHVzaD1UcnVlKQpyaD1ucC5hcnJheShyaG9zKTsgZmluPXJoW25wLmlzZmluaXRlKHJoKV0KcHJpbnQoIj09PSBDT05UUk9MIDIgUkVTVUxUID09PSIpCnByaW50KCJ1bmRlZmluZWQgc3BsaXRzOiAlZCBvZiAlZCIlKGludCgofm5wLmlzZmluaXRlKHJoKSkuc3VtKCkpLE5TKSkKcHJpbnQoIm1lZGlhbiByaG8gb3ZlciBmaW5pdGU6ICUrLjRmIiVmbG9hdChucC5tZWRpYW4oZmluKSkgaWYgZmluLnNpemUgZWxzZSAiQUxMIFVOREVGSU5FRCIpCnByaW50KCJyYW5nZTogJSsuNGYgdG8gJSsuNGYiJShmbG9hdChmaW4ubWluKCkpLGZsb2F0KGZpbi5tYXgoKSkpIGlmIGZpbi5zaXplIGVsc2UgIiIpCnByaW50KCJWT0lEIENSSVRFUklPTiBtZWRpYW4gcmhvID4gMC41IC0+IiwgIlZPSUQiIGlmIChmaW4uc2l6ZSBhbmQgbnAubWVkaWFuKGZpbik+MC41KSBlbHNlICJOT1QgVk9JRCIpCmlmICh+bnAuaXNmaW5pdGUocmgpKS5zdW0oKT5OUy8yOiBwcmludCgiTU9SRSBUSEFOIEhBTEYgVU5ERUZJTkVEIC0+IElOQ09OQ0xVU0lWRSwgTk9UIFBBU1NFRCIpCm5wLnNhdmUoImNvbnRyb2wyX3Job3MubnB5IixyaCk7IHJlYWwudG9fY3N2KCJjb250cm9sMl9yZWFsX3Njb3Jlcy5jc3YiKQpwcmludCgiVE9UQUwiLHJvdW5kKHRpbWUudGltZSgpLXQwLDEpLCJzIikK"

In [13]:
import base64,hashlib
B=S1+S2+S3+S4+S5+S6
print("len",len(B),"want",6600)
print("md5",hashlib.md5(B.encode()).hexdigest(),"want","30a1f35d58f249c8e770e58a9270b1f0")
assert len(B)==6600 and hashlib.md5(B.encode()).hexdigest()=="30a1f35d58f249c8e770e58a9270b1f0","TRANSFER CORRUPTED"
exec(base64.b64decode(B))

len 6600 want 6600
md5 30a1f35d58f249c8e770e58a9270b1f0 want 30a1f35d58f249c8e770e58a9270b1f0
library (90834, 951) compounds 26425
samples in table 313 genes with >=1 probe 951
probes per gene: median 1 max 3
genes usable 951 of 951
cases 170 controls 133
REAL query: compounds 26425 frac exactly zero 0.3661 spread 0.3958
  split  1 rho +0.2724
  split  2 rho +0.0562
  split  3 rho +0.4806
  split  4 rho -0.3110
  split  5 rho +0.1062
  split  6 rho -0.1380
  split  7 rho +0.2519
  split  8 rho -0.5059
  split  9 rho +0.0340
  split 10 rho +0.5055
  split 11 rho -0.1431
  split 12 rho -0.2802
  split 13 rho +0.1922
  split 14 rho +0.4183
  split 15 rho +0.1415
  split 16 rho +0.2554
  split 17 rho -0.0046
  split 18 rho -0.3105
  split 19 rho +0.1754
  split 20 rho +0.2470
=== CONTROL 2 RESULT ===
undefined splits: 0 of 20
median rho over finite: +0.1238
range: -0.5059 to +0.5055
VOID CRITERION median rho > 0.5 -> NOT VOID
TOTAL 311.0 s


In [14]:
import csv
from collections import Counter
ros2={r["gsm"]:r["group"] for r in csv.DictReader(open("GSE57338_roster.csv",encoding="utf-8"))}
H=set(samples); R=set(ros2)
print("roster rows",len(ros2),"roster groups",Counter(ros2.values()))
print("table samples",len(H),"roster keys",len(R))
print("in table not in roster:",len(H-R),sorted(H-R)[:12])
print("in roster not in table:",len(R-H),sorted(R-H)[:12])
print("matched groups",Counter(ros2[s] for s in samples if s in ros2))

roster rows 303 roster groups Counter({'case': 170, 'control': 133})
table samples 313 roster keys 303
in table not in roster: 10 ['GSM1380115', 'GSM1380116', 'GSM1380117', 'GSM1380118', 'GSM1380119', 'GSM1380120', 'GSM1380121', 'GSM1380122', 'GSM1380123', 'GSM1380124']
in roster not in table: 0 []
matched groups Counter({'case': 170, 'control': 133})


In [15]:
import csv,base64
f=open("GSE57338_roster.csv","w",newline="",encoding="utf-8")
w=csv.writer(f)
w.writerow(["gsm","group","characteristics"])
for g in ros: w.writerow([g,ros[g] or "unassigned"," ; ".join(meta[g])])
f.close()
n=sum(1 for _ in open("GSE57338_roster.csv",encoding="utf-8"))-1
print("roster rows now",n,"want",len(ros))
assert n==len(ros),"ROSTER STILL SHORT"
exec(base64.b64decode(B))

roster rows now 313 want 303


AssertionError: ROSTER STILL SHORT

In [16]:
import csv,base64
ros={g:cls(fl) for g,fl in meta.items()}
print("meta entries",len(meta),"ros entries",len(ros))
fh=open("GSE57338_roster.csv","w",newline="",encoding="utf-8")
w=csv.writer(fh)
w.writerow(["gsm","group","characteristics"])
for g in ros: w.writerow([g,ros[g] or "unassigned"," ; ".join(meta[g])])
fh.close()
rec=list(csv.DictReader(open("GSE57338_roster.csv",encoding="utf-8")))
print("records read back",len(rec))
assert len(rec)==len(meta),"ROSTER STILL WRONG"
exec(base64.b64decode(B))

meta entries 313 ros entries 313
records read back 313
library (90834, 951) compounds 26425
samples in table 313 genes with >=1 probe 951
probes per gene: median 1 max 3
genes usable 951 of 951
cases 177 controls 136
REAL query: compounds 26425 frac exactly zero 0.3469 spread 0.4055
  split  1 rho +0.3163
  split  2 rho +0.1532
  split  3 rho +0.3926
  split  4 rho +0.1876
  split  5 rho -0.1356
  split  6 rho -0.4186
  split  7 rho -0.1517
  split  8 rho -0.4731
  split  9 rho +0.0456
  split 10 rho +0.4091
  split 11 rho +0.2766
  split 12 rho -0.3454
  split 13 rho -0.2103
  split 14 rho +0.5384
  split 15 rho -0.2117
  split 16 rho +0.1635
  split 17 rho +0.3632
  split 18 rho +0.0282
  split 19 rho +0.2353
  split 20 rho +0.0400
=== CONTROL 2 RESULT ===
undefined splits: 0 of 20
median rho over finite: +0.0994
range: -0.4731 to +0.5384
VOID CRITERION median rho > 0.5 -> NOT VOID
TOTAL 302.4 s
